# 🚀 Agentic4API — Évaluation RAG
## Gemini Pro + Pinecone · 150 APIs · 100 questions

Ce notebook évalue les performances du système RAG **Agentic4API** :

| Paramètre | Valeur |
|-----------|--------|
| **Agent** | Gemini Pro + Pinecone |
| **Catalogue** | 150 APIs OpenAPI 3.0 |
| **Golden dataset** | 100 questions |
| **Métriques** | Precision@K · Recall@K · MRR |
| **Catégories** | simple · faux_positif · versioning · breaking_change · multi_api · endpoint_precise · negative · use_case · authorization |

> **Colab** : Runtime → Run all  
> Les résultats sont téléchargés automatiquement à la fin.

## ⚙️ Cellule 1 — Installation des dépendances

In [ ]:
# Décommenter si nécessaire (Colab installe numpy/pandas par défaut)
# !pip install -q requests pandas numpy matplotlib

import numpy as np
import pandas as pd
print(f'numpy  : {np.__version__}')
print(f'pandas : {pd.__version__}')
print('✅ Dépendances OK')

numpy  : 1.26.4
pandas : 2.2.2
✅ Dépendances OK


## 🔧 Cellule 2 — Configuration
URL du webhook n8n et liste complète des 150 APIs.

In [ ]:
AGENT_URL  = "https://nexdigital.app.n8n.cloud/webhook/fb81ea57-3ffa-4e60-b8b2-a7c83326dbbb/chat"
AGENT_NAME = "Gemini Pro + Pinecone"
TOP_K      = 4

# ── Liste complète des 150 APIs ──────────────────────────────────────────────
ALL_APIS = [
    # E-Commerce
    "cart-api", "wishlist-api", "checkout-api", "order-api-v1", "order-api-v2",
    "order-api-v3", "order-api-v4", "product-api-v1", "product-api-v2",
    "product-api-v3", "product-api-v4", "product-catalog-api", "catalog-api",
    "pricing-api", "price-list-api", "discount-api", "promotion-api",
    "search-api", "search-api-v2", "search-api-v3",
    "review-api", "rating-api", "bundle-api", "gift-card-api", "pre-order-api",
    "recommendation-api", "stock-reservation-api",
    # Logistique / Supply Chain
    "shipping-api", "shipping-api-v2", "delivery-api", "return-api",
    "inventory-api", "inventory-api-v1", "inventory-api-v2",
    "warehouse-api", "logistics-tracking-api", "store-locator-api",
    "supplier-api", "purchase-order-api", "demand-forecast-api", "quality-api",
    # Finance
    "payment-api", "payment-api-v1", "payment-api-v2", "payment-api-v3",
    "billing-api", "billing-api-v2", "billing-api-v3",
    "invoice-api", "invoice-api-v2", "invoice-api-v3",
    "wallet-api", "tax-api", "escrow-api", "payout-api", "subscription-api",
    # Identity & Access
    "user-api", "user-api-v1", "user-api-v3",
    "auth-api", "auth-api-v1", "sso-api", "mfa-api", "session-api",
    "account-api", "profile-api",
    # CRM & Marketing
    "customer-profile-api", "customer-profile-api-v2",
    "crm-contact-api", "lead-api", "campaign-api",
    "loyalty-points-api", "segmentation-api", "referral-api",
    # Communication
    "notification-api", "notification-api-v2", "notification-api-v3",
    "email-api", "sms-api", "push-api", "messaging-api", "alert-api",
    "webhook-api", "live-chat-api", "chatbot-api", "contact-form-api",
    # Analytics & BI
    "analytics-api", "analytics-api-v2", "analytics-api-v3",
    "reporting-api", "reporting-api-v2", "metrics-api",
    "event-tracking-api", "ab-testing-api", "data-export-api",
    "attribution-api", "cohort-api", "customer-journey-api", "report-api",
    # RH
    "hr-api", "employee-api", "payroll-api", "leave-api", "recruitment-api",
    "performance-api", "training-api", "expense-api", "time-tracking-api",
    "benefits-api", "onboarding-api", "org-api",
    # Platform & Infrastructure
    "health-api", "audit-log-api", "gdpr-api", "api-key-api",
    "rate-limit-api", "config-api", "feature-flag-api", "log-api",
    "cache-api", "queue-api", "file-storage-api", "media-api",
    "media-processing-api", "document-api", "cdn-api", "dns-api",
    "secret-api", "integration-api", "workflow-api", "event-api",
    "permission-api", "encryption-api",
    # Support Client
    "ticket-api", "knowledge-base-api", "survey-api",
    # Localisation
    "geolocation-api", "localization-api", "translation-api",
    # Ops / Projets
    "task-api", "project-api", "team-api", "comment-api",
    "contract-api", "tag-api",
    # Autres
    "address-api", "contact-api", "appointment-api", "calendar-api",
    "accessibility-api", "ab-testing-api",
]
# Dédupliquer (ab-testing-api apparaît deux fois)
ALL_APIS = list(dict.fromkeys(ALL_APIS))

print(f"✅ {len(ALL_APIS)} APIs dans le catalogue")
print(f"✅ Agent cible : {AGENT_NAME}")

✅ 150 APIs dans le catalogue
✅ Agent cible : Gemini Pro + Pinecone


## 📋 Cellule 3 — Golden Dataset (100 questions)

**Distribution par catégorie :**

| Catégorie | N | Description |
|-----------|---|-------------|
| `simple` | 15 | 1 API évidente |
| `faux_positif` | 20 | APIs sémantiquement proches mais ≠ |
| `versioning` | 11 | Versions coexistantes, migrations |
| `breaking_change` | 4 | Différences précises entre versions |
| `multi_api` | 15 | Orchestration de plusieurs APIs |
| `endpoint_precise` | 10 | Endpoint exact demandé |
| `negative` | 10 | API inexistante dans le catalogue |
| `use_case` | 10 | Cas d'usage métier complet |
| `authorization` | 5 | Scopes, tokens, SSO |


In [ ]:
GOLDEN_DATASET = [

    # ── SIMPLE (15 questions) ────────────────────────────────────────────────
    {"id":"Q001","question":"Quelle API utiliser pour créer une nouvelle commande ?",
     "expected_apis":["order-api-v4"],"difficulty":"easy","category":"simple",
     "expected_answer":"Order API v4 — POST /v4/orders"},

    {"id":"Q002","question":"Comment récupérer le profil d'un utilisateur connecté ?",
     "expected_apis":["user-api"],"difficulty":"easy","category":"simple",
     "expected_answer":"User API — GET /v2/users/{id}"},

    {"id":"Q003","question":"Quelle API permet de rechercher des produits par mot-clé ?",
     "expected_apis":["search-api-v3"],"difficulty":"easy","category":"simple",
     "expected_answer":"Search API v3 — GET /v3/search?q={keyword}"},

    {"id":"Q004","question":"Je veux vérifier le stock disponible pour un produit.",
     "expected_apis":["inventory-api"],"difficulty":"easy","category":"simple",
     "expected_answer":"Inventory API v3 — GET /v3/inventory/{productId}"},

    {"id":"Q005","question":"Comment obtenir le prix actuel d'un produit ?",
     "expected_apis":["pricing-api"],"difficulty":"easy","category":"simple",
     "expected_answer":"Pricing API — GET /v1/pricing/product/{productId}"},

    {"id":"Q006","question":"Quelle API gère l'authentification des utilisateurs ?",
     "expected_apis":["auth-api"],"difficulty":"easy","category":"simple",
     "expected_answer":"Auth API v2 — POST /v2/auth/login"},

    {"id":"Q007","question":"Comment ajouter un article au panier d'un utilisateur ?",
     "expected_apis":["cart-api"],"difficulty":"easy","category":"simple",
     "expected_answer":"Cart API — POST /v1/cart/{userId}/items"},

    {"id":"Q008","question":"Quelle API envoie une notification push à un utilisateur ?",
     "expected_apis":["notification-api-v3"],"difficulty":"easy","category":"simple",
     "expected_answer":"Notification API v3 — POST /v3/notifications/send avec channel=push"},

    {"id":"Q009","question":"Comment suivre la livraison d'un colis ?",
     "expected_apis":["shipping-api-v2"],"difficulty":"easy","category":"simple",
     "expected_answer":"Shipping API v2 — GET /v2/shipping/{id}/live-track"},

    {"id":"Q010","question":"Quelle API permet de laisser un avis sur un produit ?",
     "expected_apis":["review-api"],"difficulty":"easy","category":"simple",
     "expected_answer":"Review API — POST /v1/reviews"},

    {"id":"Q011","question":"Comment générer une facture après une commande payée ?",
     "expected_apis":["invoice-api-v3"],"difficulty":"easy","category":"simple",
     "expected_answer":"Invoice API v3 — POST /v3/invoices"},

    {"id":"Q012","question":"Quelle API permet de créer un bon de commande fournisseur ?",
     "expected_apis":["purchase-order-api"],"difficulty":"easy","category":"simple",
     "expected_answer":"Purchase Order API — POST /v1/purchase-orders"},

    {"id":"Q013","question":"Comment créer une tâche dans le système ?",
     "expected_apis":["task-api"],"difficulty":"easy","category":"simple",
     "expected_answer":"Task API — POST /v1/tasks"},

    {"id":"Q014","question":"Quelle API gère les congés des employés ?",
     "expected_apis":["leave-api"],"difficulty":"easy","category":"simple",
     "expected_answer":"Leave API — POST /v1/leaves/request"},

    {"id":"Q015","question":"Comment calculer les taxes d'une transaction ?",
     "expected_apis":["tax-api"],"difficulty":"easy","category":"simple",
     "expected_answer":"Tax API — POST /v1/tax/calculate"},

    # ── FAUX POSITIFS (20 questions) ─────────────────────────────────────────
    {"id":"Q016","question":"Quelle est la différence entre la User API et la Customer Profile API ?",
     "expected_apis":["user-api","customer-profile-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"User API = identité technique (login, mdp, 2FA). Customer Profile API = données commerciales (segment, fidélité, achats)."},

    {"id":"Q017","question":"Je veux gérer le profil commercial d'un acheteur, pas ses credentials.",
     "expected_apis":["customer-profile-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Customer Profile API — pas User API. User API gère login/mdp."},

    {"id":"Q018","question":"Quelle est la différence entre la Notification API et la Messaging API ?",
     "expected_apis":["notification-api-v3","messaging-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Notification = unidirectionnel (système→user). Messaging = bidirectionnel (user↔user)."},

    {"id":"Q019","question":"Je veux que deux utilisateurs puissent se envoyer des messages entre eux.",
     "expected_apis":["messaging-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Messaging API — bidirectionnel user↔user. Pas Notification API."},

    {"id":"Q020","question":"Quelle différence entre Payment API et Billing API ?",
     "expected_apis":["payment-api-v3","billing-api-v3"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Payment = transaction ponctuelle. Billing = cycles récurrents (abonnements)."},

    {"id":"Q021","question":"Je veux mettre en place un abonnement mensuel avec prélèvement automatique.",
     "expected_apis":["billing-api-v3"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Billing API v3 — pas Payment API. Payment gère les transactions ponctuelles."},

    {"id":"Q022","question":"Quelle est la différence entre Invoice API et Billing API ?",
     "expected_apis":["invoice-api-v3","billing-api-v3"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Invoice = document fiscal PDF. Billing = orchestration des prélèvements."},

    {"id":"Q023","question":"Je veux générer un PDF de facture conforme à envoyer au client.",
     "expected_apis":["invoice-api-v3"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Invoice API v3 — pas Billing API. Invoice génère les documents fiscaux."},

    {"id":"Q024","question":"Quelle différence entre Alert API et Notification API ?",
     "expected_apis":["alert-api","notification-api-v3"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Alert = équipes OPS internes (PagerDuty, Slack). Notification = clients finaux."},

    {"id":"Q025","question":"Je veux alerter mon équipe technique quand un service tombe.",
     "expected_apis":["alert-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Alert API — pas Notification API. Alert cible les équipes ops."},

    {"id":"Q026","question":"Quelle différence entre Task API et Ticket API ?",
     "expected_apis":["task-api","ticket-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Task = tâche interne planifiée (projet, sprint). Ticket = demande support client."},

    {"id":"Q027","question":"Un client a un problème, je veux créer une demande de support.",
     "expected_apis":["ticket-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Ticket API — pas Task API. Ticket gère les demandes support clients."},

    {"id":"Q028","question":"Quelle est la différence entre Media API et Media Processing API ?",
     "expected_apis":["media-api","media-processing-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Media API = stockage + CDN + resize simple. Media Processing = transcodage vidéo, OCR, watermark, détection IA."},

    {"id":"Q029","question":"Je veux extraire du texte d'un PDF scanné.",
     "expected_apis":["media-processing-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Media Processing API — OCR endpoint. Pas File Storage ni Media API."},

    {"id":"Q030","question":"Quelle différence entre Catalog API et Product Catalog API ?",
     "expected_apis":["catalog-api","product-catalog-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Catalog API = collections publiées (ex: catalogue été 2026). Product Catalog API = taxonomie et catégories produits."},

    {"id":"Q031","question":"Je veux créer une collection de produits pour mon canal B2B Allemagne.",
     "expected_apis":["catalog-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Catalog API — pas Product Catalog API. Catalog gère les collections publiées par canal."},

    {"id":"Q032","question":"Quelle différence entre Report API et Reporting API ?",
     "expected_apis":["report-api","reporting-api-v2"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Report API = rapport contextuel instantané sur une ressource spécifique. Reporting API = rapports planifiés périodiques sur des populations."},

    {"id":"Q033","question":"Je veux un rapport PDF mensuel des ventes envoyé par email automatiquement.",
     "expected_apis":["reporting-api-v2"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Reporting API v2 — pas Report API. Reporting gère les rapports planifiés récurrents."},

    {"id":"Q034","question":"Quelle différence entre Event API et Event Tracking API ?",
     "expected_apis":["event-api","event-tracking-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Event API = bus d'événements système entre microservices. Event Tracking = comportements utilisateurs pour analytics."},

    {"id":"Q035","question":"Je veux tracer les clics et pages vues des utilisateurs sur mon site.",
     "expected_apis":["event-tracking-api"],"difficulty":"medium","category":"faux_positif",
     "expected_answer":"Event Tracking API — pas Event API. Event Tracking collecte les comportements utilisateurs."},

    # ── VERSIONING (15 questions) ─────────────────────────────────────────────
    {"id":"Q036","question":"Quelle est la version actuelle recommandée de l'API de commandes ?",
     "expected_apis":["order-api-v4"],"difficulty":"easy","category":"versioning",
     "expected_answer":"Order API v4 est la version actuelle et recommandée."},

    {"id":"Q037","question":"Quels sont les breaking changes entre Order API v3 et v4 ?",
     "expected_apis":["order-api-v3","order-api-v4"],"difficulty":"hard","category":"breaking_change",
     "expected_answer":"v4 : items→order_lines, total_price→amount_due, currency obligatoire, webhooks automatiques, /ship supprimé→PATCH /status"},

    {"id":"Q038","question":"J'utilise order-api-v2, vers quoi dois-je migrer ?",
     "expected_apis":["order-api-v4"],"difficulty":"medium","category":"versioning",
     "expected_answer":"Migrer vers order-api-v4 (v2 deprecated depuis juin 2022). Attention aux breaking changes v2→v3→v4."},

    {"id":"Q039","question":"Quelles APIs de paiement coexistent et lesquelles sont dépréciées ?",
     "expected_apis":["payment-api","payment-api-v1","payment-api-v2","payment-api-v3"],"difficulty":"hard","category":"versioning",
     "expected_answer":"v1 deprecated (2021), v2 deprecated (2024), v3 est la version actuelle."},

    {"id":"Q040","question":"Quelle est la différence entre payment-api-v2 et payment-api-v3 ?",
     "expected_apis":["payment-api-v2","payment-api-v3"],"difficulty":"hard","category":"breaking_change",
     "expected_answer":"v3 ajoute BNPL (3x/4x), crypto (BTC/ETH/USDC), dispute management et réconciliation bancaire."},

    {"id":"Q041","question":"Quelle version de l'User API supporte SCIM 2.0 et les groupes ?",
     "expected_apis":["user-api-v3"],"difficulty":"medium","category":"versioning",
     "expected_answer":"User API v3 — introduit SCIM 2.0, groupes natifs et provisioning IdP SSO."},

    {"id":"Q042","question":"Quelle version de l'Inventory API introduit les alertes de seuil ?",
     "expected_apis":["inventory-api-v2"],"difficulty":"medium","category":"versioning",
     "expected_answer":"Inventory API v2 — introduit les alertes de seuil et l'historique des mouvements."},

    {"id":"Q043","question":"Quelle version de la Notification API supporte WhatsApp ?",
     "expected_apis":["notification-api-v3"],"difficulty":"medium","category":"versioning",
     "expected_answer":"Notification API v3 — ajout canaux WhatsApp et in-app, triggers événementiels."},

    {"id":"Q044","question":"Quelle version de la Search API intègre un moteur LLM conversationnel ?",
     "expected_apis":["search-api-v3"],"difficulty":"medium","category":"versioning",
     "expected_answer":"Search API v3 — mode conversational (LLM) et synonymes configurables."},

    {"id":"Q045","question":"Quelle version du Product API gère les catalogues multi-canal B2B/B2C ?",
     "expected_apis":["product-api-v4"],"difficulty":"medium","category":"versioning",
     "expected_answer":"Product API v4 — multi-catalogue natif et cycle de vie lifecycle.stage."},

    {"id":"Q046","question":"Qu'est-ce qui change entre Invoice API v2 et v3 ?",
     "expected_apis":["invoice-api-v2","invoice-api-v3"],"difficulty":"hard","category":"breaking_change",
     "expected_answer":"v3 : facturation récurrente native, pénalités de retard automatiques, export comptable FEC/DATEV/SAGE."},

    {"id":"Q047","question":"Quelle version de la Billing API supporte la facturation à l'usage (metered) ?",
     "expected_apis":["billing-api-v3"],"difficulty":"medium","category":"versioning",
     "expected_answer":"Billing API v3 — billing_mode: flat | metered | hybrid et sync ERP natif."},

    {"id":"Q048","question":"Quelle est la différence entre Shipping API v1 et v2 ?",
     "expected_apis":["shipping-api","shipping-api-v2"],"difficulty":"medium","category":"breaking_change",
     "expected_answer":"v2 : multi-colis, tracking GPS temps réel (SSE), CO2 calculé, support ZPL."},

    {"id":"Q049","question":"Quelle version de l'Analytics API inclut des prédictions ML ?",
     "expected_apis":["analytics-api-v3"],"difficulty":"medium","category":"versioning",
     "expected_answer":"Analytics API v3 — endpoint /predict avec modèles prophet/lstm et alertes sur métriques."},

    {"id":"Q050","question":"J'utilise auth-api-v1 à base de sessions serveur, vers quoi migrer ?",
     "expected_apis":["auth-api"],"difficulty":"medium","category":"versioning",
     "expected_answer":"Migrer vers Auth API v2 — JWT stateless, refresh tokens et OAuth2."},

    # ── MULTI-API (15 questions) ──────────────────────────────────────────────
    {"id":"Q051","question":"Comment créer une commande, vérifier le stock et envoyer une confirmation client ?",
     "expected_apis":["order-api-v4","inventory-api","notification-api-v3"],"difficulty":"hard","category":"multi_api",
     "expected_answer":"Inventory pour le stock → Order v4 pour la commande → Notification v3 pour la confirmation."},

    {"id":"Q052","question":"Comment implémenter un tunnel d'achat complet ?",
     "expected_apis":["cart-api","pricing-api","order-api-v4","payment-api-v3","notification-api-v3","shipping-api-v2"],"difficulty":"hard","category":"multi_api",
     "expected_answer":"Cart → Pricing → Order v4 → Payment v3 → Notification v3 → Shipping v2."},

    {"id":"Q053","question":"Je veux créer un nouveau collaborateur : dossier RH, accès système et email de bienvenue.",
     "expected_apis":["hr-api","user-api-v3","onboarding-api","notification-api-v3"],"difficulty":"hard","category":"multi_api",
     "expected_answer":"HR API pour le dossier → User API v3 pour les accès → Onboarding API → Notification v3."},

    {"id":"Q054","question":"Comment construire une page produit complète avec prix, stock, avis et recommandations ?",
     "expected_apis":["product-api-v4","pricing-api","inventory-api","review-api","recommendation-api"],"difficulty":"hard","category":"multi_api",
     "expected_answer":"Product v4 + Pricing + Inventory + Review + Recommendation APIs."},

    {"id":"Q055","question":"Je veux notifier automatiquement un client quand sa commande est expédiée.",
     "expected_apis":["order-api-v4","shipping-api-v2","notification-api-v3"],"difficulty":"hard","category":"multi_api",
     "expected_answer":"Order v4 (webhook statut) → Shipping v2 (tracking) → Notification v3 (alerte client)."},

    {"id":"Q056","question":"Comment mettre en place un programme de fidélité après chaque achat ?",
     "expected_apis":["order-api-v4","loyalty-points-api","notification-api-v3"],"difficulty":"medium","category":"multi_api",
     "expected_answer":"Order v4 (achat confirmé) → Loyalty Points (attribution) → Notification v3 (confirmation points)."},

    {"id":"Q057","question":"Comment gérer un retour produit : demande, validation, remboursement et email ?",
     "expected_apis":["return-api","payment-api-v3","invoice-api-v3","notification-api-v3"],"difficulty":"hard","category":"multi_api",
     "expected_answer":"Return API → Payment v3 (remboursement) → Invoice v3 (avoir) → Notification v3 (email)."},

    {"id":"Q058","question":"Je veux analyser les ventes et envoyer un rapport hebdomadaire à l'équipe.",
     "expected_apis":["analytics-api-v3","reporting-api-v2","notification-api-v3"],"difficulty":"medium","category":"multi_api",
     "expected_answer":"Analytics v3 (métriques) → Reporting v2 (planifié) → Notification v3 (envoi équipe)."},

    {"id":"Q059","question":"Comment créer un contrat, le faire signer et archiver le document ?",
     "expected_apis":["contract-api","document-api"],"difficulty":"medium","category":"multi_api",
     "expected_answer":"Contract API (cycle de vie) → Document API (signature eIDAS + archivage)."},

    {"id":"Q060","question":"Onboarding client B2B : créer le compte organisation, inviter des membres, configurer la facturation.",
     "expected_apis":["account-api","user-api-v3","billing-api-v3"],"difficulty":"hard","category":"multi_api",
     "expected_answer":"Account API (organisation) → User API v3 (membres) → Billing v3 (abonnement)."},

    {"id":"Q061","question":"Je veux appliquer un code promo sur une commande et recalculer le prix final.",
     "expected_apis":["discount-api","pricing-api","order-api-v4"],"difficulty":"medium","category":"multi_api",
     "expected_answer":"Discount API (code promo) → Pricing API (calcul remise) → Order v4 (commande)."},

    {"id":"Q062","question":"Comment détecter une rupture de stock imminente et commander automatiquement ?",
     "expected_apis":["inventory-api","demand-forecast-api","purchase-order-api"],"difficulty":"hard","category":"multi_api",
     "expected_answer":"Inventory (alertes seuil) → Demand Forecast (prévisions) → Purchase Order (réapprovisionnement)."},

    {"id":"Q063","question":"Je veux créer un A/B test sur ma page de résultats de recherche.",
     "expected_apis":["ab-testing-api","search-api-v3","analytics-api-v3"],"difficulty":"hard","category":"multi_api",
     "expected_answer":"A/B Testing (expérience) → Search v3 (résultats par variante) → Analytics v3 (mesure conversions)."},

    {"id":"Q064","question":"Comment segmenter mes clients et leur envoyer une campagne marketing ciblée ?",
     "expected_apis":["segmentation-api","campaign-api","notification-api-v3"],"difficulty":"medium","category":"multi_api",
     "expected_answer":"Segmentation (règles) → Campaign API (campagne) → Notification v3 (envoi multicanal)."},

    {"id":"Q065","question":"Construire un flux de paiement marketplace : encaisser l'acheteur, retenir en séquestre, reverser au vendeur.",
     "expected_apis":["payment-api-v3","escrow-api","payout-api"],"difficulty":"hard","category":"multi_api",
     "expected_answer":"Payment v3 (encaissement) → Escrow (séquestre) → Payout (reversement vendeur)."},

    # ── ENDPOINT PRÉCIS (10 questions) ───────────────────────────────────────
    {"id":"Q066","question":"Quel endpoint utiliser pour mettre à jour partiellement le statut d'une commande v4 ?",
     "expected_apis":["order-api-v4"],"difficulty":"medium","category":"endpoint_precise",
     "expected_answer":"PATCH /v4/orders/{id}/status"},

    {"id":"Q067","question":"Quel endpoint génère une description produit par IA ?",
     "expected_apis":["product-api-v4"],"difficulty":"medium","category":"endpoint_precise",
     "expected_answer":"POST /v4/products/{id}/ai-description"},

    {"id":"Q068","question":"Quel endpoint permet de rejouer des événements passés après un incident ?",
     "expected_apis":["event-api"],"difficulty":"medium","category":"endpoint_precise",
     "expected_answer":"POST /v1/events/replay"},

    {"id":"Q069","question":"Quel endpoint retourne les créneaux de livraison disponibles ?",
     "expected_apis":["delivery-api"],"difficulty":"easy","category":"endpoint_precise",
     "expected_answer":"GET /v1/delivery/slots"},

    {"id":"Q070","question":"Quel endpoint invalide toutes les sessions actives d'un utilisateur ?",
     "expected_apis":["session-api"],"difficulty":"medium","category":"endpoint_precise",
     "expected_answer":"DELETE /v1/sessions/{userId}/revoke-all"},

    {"id":"Q071","question":"Quel endpoint exporte une facture au format comptable FEC ?",
     "expected_apis":["invoice-api-v3"],"difficulty":"medium","category":"endpoint_precise",
     "expected_answer":"GET /v3/invoices/{id}/accounting-export?format=FEC"},

    {"id":"Q072","question":"Quel endpoint crée une règle de déclenchement automatique de notification ?",
     "expected_apis":["notification-api-v3"],"difficulty":"medium","category":"endpoint_precise",
     "expected_answer":"POST /v3/notifications/triggers"},

    {"id":"Q073","question":"Quel endpoint donne les prédictions ML de demande pour les 30 prochains jours ?",
     "expected_apis":["demand-forecast-api"],"difficulty":"medium","category":"endpoint_precise",
     "expected_answer":"POST /v1/forecast/demand avec horizon_days=30"},

    {"id":"Q074","question":"Quel endpoint liste les sessions suspectes détectées ?",
     "expected_apis":["session-api"],"difficulty":"medium","category":"endpoint_precise",
     "expected_answer":"GET /v1/sessions/suspicious"},

    {"id":"Q075","question":"Quel endpoint du Billing API synchronise les données avec SAP ?",
     "expected_apis":["billing-api-v3"],"difficulty":"hard","category":"endpoint_precise",
     "expected_answer":"POST /v3/billing/erp-sync/{subscriptionId} avec erp_system=sap"},

    # ── NÉGATIFS (10 questions) ───────────────────────────────────────────────
    {"id":"Q076","question":"Quelle API gère les paiements en cryptomonnaies Bitcoin ?",
     "expected_apis":[],"difficulty":"medium","category":"negative",
     "expected_answer":"Payment API v3 supporte Bitcoin (BTC), Ethereum (ETH) et USDC via method=bitcoin|ethereum|usdc"},

    {"id":"Q077","question":"Y a-t-il une API de reconnaissance faciale dans le catalogue ?",
     "expected_apis":[],"difficulty":"easy","category":"negative",
     "expected_answer":"Non, il n'existe pas d'API de reconnaissance faciale dans le catalogue."},

    {"id":"Q078","question":"Quelle API gère les NFTs et tokens blockchain ?",
     "expected_apis":[],"difficulty":"easy","category":"negative",
     "expected_answer":"Non, il n'existe pas d'API blockchain ou NFT dans le catalogue."},

    {"id":"Q079","question":"Est-ce qu'il existe une API de livraison par drone ?",
     "expected_apis":[],"difficulty":"easy","category":"negative",
     "expected_answer":"Non, il n'existe pas d'API de livraison par drone dans le catalogue."},

    {"id":"Q080","question":"Quelle API gère les contrats intelligents Ethereum (smart contracts) ?",
     "expected_apis":[],"difficulty":"medium","category":"negative",
     "expected_answer":"Non, il n'existe pas d'API de smart contracts dans le catalogue."},

    {"id":"Q081","question":"Y a-t-il une API de traduction en temps réel par IA générative ?",
     "expected_apis":["translation-api"],"difficulty":"medium","category":"negative",
     "expected_answer":"Oui, Translation API — POST /v1/translate (NMT neural machine translation)."},

    {"id":"Q082","question":"Quelle API gère les réseaux sociaux (Twitter, LinkedIn) ?",
     "expected_apis":[],"difficulty":"easy","category":"negative",
     "expected_answer":"Non, il n'existe pas d'API de gestion de réseaux sociaux dans le catalogue."},

    {"id":"Q083","question":"Y a-t-il une API de prise de rendez-vous médical ?",
     "expected_apis":["appointment-api"],"difficulty":"medium","category":"negative",
     "expected_answer":"Appointment API existe mais pour les conseillers/techniciens, pas pour le médical spécifiquement."},

    {"id":"Q084","question":"Quelle API gère les abonnements Netflix-like avec épisodes et contenus ?",
     "expected_apis":[],"difficulty":"medium","category":"negative",
     "expected_answer":"Non, pas d'API de streaming de contenu dans le catalogue. Subscription API gère les abonnements SaaS uniquement."},

    {"id":"Q085","question":"Est-ce qu'il y a une API de reconnaissance vocale (speech-to-text) ?",
     "expected_apis":[],"difficulty":"easy","category":"negative",
     "expected_answer":"Non, il n'existe pas d'API de speech-to-text dans le catalogue."},

    # ── AUTORISATION / SCOPES (5 questions) ──────────────────────────────────
    {"id":"Q086","question":"Comment obtenir un JWT pour appeler l'Order API v4 ?",
     "expected_apis":["auth-api","order-api-v4"],"difficulty":"medium","category":"authorization",
     "expected_answer":"Auth API v2 — POST /v2/auth/login pour obtenir le JWT, puis X-API-Key dans le header."},

    {"id":"Q087","question":"Quels scopes OAuth2 faut-il pour accéder aux données salariales ?",
     "expected_apis":["employee-api"],"difficulty":"hard","category":"authorization",
     "expected_answer":"Employee API — scope hr:payroll requis via OAuth2 client_credentials."},

    {"id":"Q088","question":"Comment configurer le SSO avec Azure AD pour les utilisateurs internes ?",
     "expected_apis":["sso-api"],"difficulty":"hard","category":"authorization",
     "expected_answer":"SSO API — POST /v1/sso/providers avec type=oidc et metadata_url Azure AD."},

    {"id":"Q089","question":"Comment activer le 2FA TOTP pour un utilisateur ?",
     "expected_apis":["mfa-api"],"difficulty":"medium","category":"authorization",
     "expected_answer":"MFA API — POST /v1/mfa/enroll avec factor_type=totp."},

    {"id":"Q090","question":"Comment vérifier si un utilisateur a le droit de modifier une commande ?",
     "expected_apis":["permission-api"],"difficulty":"medium","category":"authorization",
     "expected_answer":"Permission API — POST /v1/check avec resource=orders et action=write."},

    # ── CAS D'USAGE MÉTIER (10 questions) ─────────────────────────────────────
    {"id":"Q091","question":"Je veux construire un tableau de bord analytics avec alertes automatiques sur les KPIs.",
     "expected_apis":["analytics-api-v3"],"difficulty":"medium","category":"use_case",
     "expected_answer":"Analytics API v3 — dashboards avec ACL + endpoint /alerts pour les seuils."},

    {"id":"Q092","question":"Comment implémenter des webhooks pour être notifié des événements de paiement ?",
     "expected_apis":["webhook-api","payment-api-v3"],"difficulty":"medium","category":"use_case",
     "expected_answer":"Webhook API pour s'abonner + Payment v3 déclenche les événements."},

    {"id":"Q093","question":"Je veux construire un chatbot de support client avec escalade vers un agent humain.",
     "expected_apis":["chatbot-api","live-chat-api","ticket-api"],"difficulty":"hard","category":"use_case",
     "expected_answer":"Chatbot API (bot) → Live Chat API (escalade humain) → Ticket API (suivi)."},

    {"id":"Q094","question":"Comment gérer le cycle de vie complet d'un contrat fournisseur ?",
     "expected_apis":["contract-api","supplier-api"],"difficulty":"medium","category":"use_case",
     "expected_answer":"Supplier API (fournisseur) + Contract API (rédaction, signature, renouvellement)."},

    {"id":"Q095","question":"Je veux mesurer l'impact d'une campagne marketing sur les ventes.",
     "expected_apis":["campaign-api","analytics-api-v3","attribution-api"],"difficulty":"hard","category":"use_case",
     "expected_answer":"Campaign API (campagne) + Analytics v3 (métriques) + Attribution API (modèles first/last-touch)."},

    {"id":"Q096","question":"Comment construire un programme de parrainage avec récompenses ?",
     "expected_apis":["referral-api","loyalty-points-api","notification-api-v3"],"difficulty":"medium","category":"use_case",
     "expected_answer":"Referral API (codes) + Loyalty Points (récompenses) + Notification v3 (alertes)."},

    {"id":"Q097","question":"Je veux analyser le parcours client de la découverte à l'achat.",
     "expected_apis":["customer-journey-api","event-tracking-api","attribution-api"],"difficulty":"hard","category":"use_case",
     "expected_answer":"Event Tracking (touchpoints) + Customer Journey API (reconstruction) + Attribution (canaux)."},

    {"id":"Q098","question":"Comment mettre en place la conformité RGPD pour les données clients ?",
     "expected_apis":["gdpr-api","customer-profile-api-v2"],"difficulty":"medium","category":"use_case",
     "expected_answer":"GDPR API (droit à l'oubli, portabilité) + Customer Profile v2 (consentements)."},

    {"id":"Q099","question":"Je veux déployer progressivement une nouvelle fonctionnalité à 10% des utilisateurs.",
     "expected_apis":["feature-flag-api"],"difficulty":"easy","category":"use_case",
     "expected_answer":"Feature Flag API — créer un flag avec rollout_percent=10."},

    {"id":"Q100","question":"Comment calculer les émissions CO2 de mes expéditions et les compenser ?",
     "expected_apis":["shipping-api-v2"],"difficulty":"medium","category":"use_case",
     "expected_answer":"Shipping API v2 — champ co2_kg calculé automatiquement sur chaque expédition."},
]

print(f'✅ {len(GOLDEN_DATASET)} questions dans le golden dataset')

# Distribution
from collections import Counter
cats  = Counter(q['category']   for q in GOLDEN_DATASET)
diffs = Counter(q['difficulty'] for q in GOLDEN_DATASET)
print('\nCatégories :', dict(sorted(cats.items())))
print('Difficultés :', dict(sorted(diffs.items())))

✅ 100 questions dans le golden dataset

Catégories : {'authorization': 5, 'breaking_change': 4, 'endpoint_precise': 10, 'faux_positif': 20, 'multi_api': 15, 'negative': 10, 'simple': 15, 'use_case': 10, 'versioning': 11}
Difficultés : {'easy': 23, 'hard': 20, 'medium': 57}


## 🛠️ Cellule 4 — Fonctions d'évaluation

- `call_agent()` : envoie une question au webhook n8n avec un `sessionId` unique
- `extract_apis()` : détecte les APIs mentionnées dans la réponse
- `precision_at_k()`, `recall_at_k()`, `mrr()` : métriques RAG standard

In [ ]:
import re
import time, requests, uuid

def call_agent(url, question, timeout=60):
    try:
        session_id = str(uuid.uuid4())
        r = requests.post(
            url,
            json={"action": "sendMessage",
                  "chatInput": question,
                  "sessionId": session_id},
            headers={"Content-Type": "application/json"},
            timeout=timeout
        )
        r.raise_for_status()
        data = r.json()
        answer = data.get("output", data.get("response", data.get("text", "")))
        if isinstance(answer, list):
            answer = " ".join([
                a.get("text","") if isinstance(a,dict) else str(a)
                for a in answer
            ])
        contexts = data.get("contexts", data.get("sources", [answer]))
        if isinstance(contexts, str): contexts = [contexts]
        if not contexts:              contexts = [answer]
        return {"answer": str(answer), "contexts": contexts, "error": None}
    except Exception as e:
        return {"answer": "", "contexts": [""], "error": str(e)}


def extract_apis(answer: str) -> list:
    """Extrait UNIQUEMENT les APIs citées explicitement — pas les versions dérivées."""
    a = answer.lower()
    found = []
    for api in ALL_APIS:
        if api in a or api.replace("-", " ") in a:
            found.append(api)
    return list(dict.fromkeys(found))


def is_match(retrieved_api: str, expected_api: str) -> bool:
    """
    Un hit est valide si :
    - correspondance exacte  : "billing-api-v3" == "billing-api-v3"
    - version parente citée  : "billing-api" compte pour "billing-api-v3"
    - version plus récente   : "billing-api-v3" compte pour "billing-api"
    """
    if retrieved_api == expected_api:
        return True
    # Extraire la base (sans -vN)
    base_retrieved = re.sub(r'-v\d+$', '', retrieved_api)
    base_expected  = re.sub(r'-v\d+$', '', expected_api)
    return base_retrieved == base_expected


def precision_at_k(retrieved: list, expected: list, k: int) -> float:
    if not expected:
        return 1.0 if not retrieved else 0.0
    hits = sum(
        1 for a in retrieved[:k]
        if any(is_match(a, e) for e in expected)
    )
    return hits / k if k else 0.0


def recall_at_k(retrieved: list, expected: list, k: int) -> float:
    if not expected:
        return 1.0
    hits = sum(
        1 for e in expected
        if any(is_match(r, e) for r in retrieved[:k])
    )
    return hits / len(expected)


def mrr(retrieved: list, expected: list) -> float:
    if not expected:
        return 1.0
    for i, r in enumerate(retrieved):
        if any(is_match(r, e) for e in expected):
            return 1.0 / (i + 1)
    return 0.0


print("✅ Fonctions OK — scoring avec is_match() famille-aware")

✅ Fonctions OK — scoring avec is_match() famille-aware


## 🚀 Cellule 5 — Évaluation complète

**⏱ Durée estimée** : ~3 min (100 questions × 1.5s pause + latence agent)

> La cellule affiche en temps réel : `Precision@K`, `Recall@K`, `MRR` et la latence pour chaque question.

In [ ]:
results = []
TOTAL   = len(GOLDEN_DATASET)

print(f"\n🚀 Évaluation {AGENT_NAME} — {TOTAL} questions\n")

for i, item in enumerate(GOLDEN_DATASET):
    print(f"[{i+1:3}/{TOTAL}] {item['id']}  ({item['category']})...", end="  ")

    t0   = time.time()
    resp = call_agent(AGENT_URL, item["question"])
    lat  = round(time.time() - t0, 2)

    retrieved = extract_apis(resp["answer"])
    p  = precision_at_k(retrieved, item["expected_apis"], TOP_K)
    r  = recall_at_k(retrieved,   item["expected_apis"], TOP_K)
    m  = mrr(retrieved,           item["expected_apis"])

    results.append({
        **item,
        "retrieved_apis": retrieved,
        "answer":   resp["answer"],
        "contexts": resp["contexts"],
        "precision": round(p, 3),
        "recall":    round(r, 3),
        "mrr":       round(m, 3),
        "latency":   lat,
        "error":     resp["error"],
    })

    status = "✅" if m > 0 or not item["expected_apis"] else "❌"
    print(f"{status}  P={p:.2f}  R={r:.2f}  MRR={m:.2f}  ({lat}s)")

    # Pause pour éviter le rate-limiting
    if i < TOTAL - 1:
        time.sleep(1.5)

print(f"\n✅ Évaluation terminée — {TOTAL} questions traitées")

## 📊 Cellule 6 — Métriques et tableaux de résultats

Affiche les résultats :
- **Global** : MRR, Precision@K, Recall@K, latence
- **Par catégorie** : performance sur chaque type de question
- **Par difficulté** : easy / medium / hard
- **Questions ratées** : analyse des erreurs

In [ ]:
import pandas as pd

n = len(results)

# ── Métriques globales ───────────────────────────────────────────────────────
global_metrics = {
    "Agent":            AGENT_NAME,
    "Questions":        n,
    "Precision@K":      round(sum(r["precision"] for r in results) / n, 3),
    "Recall@K":         round(sum(r["recall"]    for r in results) / n, 3),
    "MRR":              round(sum(r["mrr"]       for r in results) / n, 3),
    "Latence moy. (s)": round(sum(r["latency"]   for r in results) / n, 2),
    "Erreurs":          sum(1 for r in results if r["error"]),
    "Réponses vides":   sum(1 for r in results if not r["answer"]),
    "Questions réussies (MRR>0)": sum(1 for r in results if r["mrr"] > 0 or not r["expected_apis"]),
}

print("\n" + "="*60)
print("  RÉSULTATS GLOBAUX —", AGENT_NAME)
print("="*60)
for k, v in global_metrics.items():
    print(f"  {k:<30} : {v}")

# ── Par catégorie ─────────────────────────────────────────────────────────────
cats = sorted(set(r["category"] for r in results))
rows_cat = []
for cat in cats:
    subset = [r for r in results if r["category"] == cat]
    rows_cat.append({
        "Catégorie":   cat,
        "N":           len(subset),
        "Precision@K": round(sum(r["precision"] for r in subset) / len(subset), 3),
        "Recall@K":    round(sum(r["recall"]    for r in subset) / len(subset), 3),
        "MRR":         round(sum(r["mrr"]       for r in subset) / len(subset), 3),
        "Latence (s)": round(sum(r["latency"]   for r in subset) / len(subset), 2),
    })

df_cat = pd.DataFrame(rows_cat).set_index("Catégorie")
print("\n" + "="*60)
print("  PAR CATÉGORIE")
print("="*60)
print(df_cat.to_string())

# ── Par difficulté ────────────────────────────────────────────────────────────
diffs = sorted(set(r["difficulty"] for r in results))
rows_diff = []
for diff in diffs:
    subset = [r for r in results if r["difficulty"] == diff]
    rows_diff.append({
        "Difficulté":  diff,
        "N":           len(subset),
        "Precision@K": round(sum(r["precision"] for r in subset) / len(subset), 3),
        "Recall@K":    round(sum(r["recall"]    for r in subset) / len(subset), 3),
        "MRR":         round(sum(r["mrr"]       for r in subset) / len(subset), 3),
        "Latence (s)": round(sum(r["latency"]   for r in subset) / len(subset), 2),
    })

df_diff = pd.DataFrame(rows_diff).set_index("Difficulté")
print("\n" + "="*60)
print("  PAR DIFFICULTÉ")
print("="*60)
print(df_diff.to_string())

# ── Questions ratées ──────────────────────────────────────────────────────────
failed = [r for r in results if r["mrr"] == 0 and r["expected_apis"]]
print(f"\n{'='*60}")
print(f"  QUESTIONS RATÉES (MRR=0) : {len(failed)}/{n}")
print("="*60)
for r in failed:
    print(f"  {r['id']} | {r['category']:<20} | {r['question'][:55]}...")
    print(f"           attendu  : {r['expected_apis']}")
    print(f"           obtenu   : {r['retrieved_apis']}")
    print()

## 📈 Cellule 7 — Dashboard de visualisation

Génère un dashboard **matplotlib** avec :
- **KPI cards** : MRR · Recall@K · Precision@K · Latence · Réussite
- **Barres** : MRR par catégorie
- **Barres** : MRR par difficulté
- **Pie** : ratio succès / échec

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Données depuis les résultats ─────────────────────────────────────────────
cat_names  = [r["Catégorie"]   for r in rows_cat]
cat_mrr    = [r["MRR"]         for r in rows_cat]
cat_prec   = [r["Precision@K"] for r in rows_cat]

diff_names = [r["Difficulté"]  for r in rows_diff]
diff_mrr   = [r["MRR"]         for r in rows_diff]

g_mrr   = global_metrics["MRR"]
g_prec  = global_metrics["Precision@K"]
g_rec   = global_metrics["Recall@K"]
g_lat   = global_metrics["Latence moy. (s)"]
g_ok    = round(global_metrics["Questions réussies (MRR>0)"] / n, 3)

COLOR   = "#378ADD"
COLOR_D = "#185FA5"
BG      = "#f8f9fa"

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor(BG)

gs = fig.add_gridspec(3, 3, hspace=0.55, wspace=0.4)
ax_kpi  = fig.add_subplot(gs[0, :])    # Ligne KPI
ax_cat  = fig.add_subplot(gs[1, :])    # MRR par catégorie
ax_diff = fig.add_subplot(gs[2, :2])   # MRR par difficulté
ax_pie  = fig.add_subplot(gs[2, 2])    # Répartition succès/échec

# ── KPI cards ────────────────────────────────────────────────────────────────
ax_kpi.set_facecolor(BG)
ax_kpi.axis("off")
kpis = [
    ("MRR",          f"{g_mrr:.3f}"),
    ("Recall@K",     f"{g_rec:.3f}"),
    ("Precision@K",  f"{g_prec:.3f}"),
    ("Latence moy.", f"{g_lat}s"),
    ("Réussite",     f"{g_ok*100:.1f}%"),
]
xs = np.linspace(0.08, 0.92, len(kpis))
for x, (label, val) in zip(xs, kpis):
    box = mpatches.FancyBboxPatch(
        (x - 0.08, 0.1), 0.16, 0.78,
        boxstyle="round,pad=0.01",
        transform=ax_kpi.transAxes,
        facecolor="#E6F1FB", edgecolor=COLOR, linewidth=0.8
    )
    ax_kpi.add_patch(box)
    ax_kpi.text(x, 0.72, label, ha="center", va="center", fontsize=10,
                color="#555", transform=ax_kpi.transAxes)
    ax_kpi.text(x, 0.38, val, ha="center", va="center", fontsize=26,
                color=COLOR_D, fontweight="bold", transform=ax_kpi.transAxes)
ax_kpi.set_title(f"Résultats globaux — {AGENT_NAME}  ({n} questions)",
                  fontsize=13, fontweight="bold", color="#222", pad=10)

# ── MRR par catégorie ─────────────────────────────────────────────────────────
x = np.arange(len(cat_names))
bars = ax_cat.bar(x, cat_mrr, color=COLOR, alpha=0.85, width=0.55)
ax_cat.set_xticks(x)
ax_cat.set_xticklabels(cat_names, fontsize=9, rotation=25, ha="right")
ax_cat.set_ylim(0, 1.18)
ax_cat.set_ylabel("MRR", fontsize=10)
ax_cat.set_title("MRR par catégorie", fontsize=12, fontweight="bold", color="#333")
ax_cat.set_facecolor(BG)
ax_cat.grid(axis="y", alpha=0.3)
ax_cat.spines[["top", "right"]].set_visible(False)
for bar, val in zip(bars, cat_mrr):
    ax_cat.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8, color=COLOR_D)

# ── MRR par difficulté ────────────────────────────────────────────────────────
x2 = np.arange(len(diff_names))
bars2 = ax_diff.bar(x2, diff_mrr, color=[COLOR_D, COLOR, "#85B7EB"][:len(diff_names)],
                    alpha=0.85, width=0.45)
ax_diff.set_xticks(x2)
ax_diff.set_xticklabels(diff_names, fontsize=11)
ax_diff.set_ylim(0, 1.15)
ax_diff.set_ylabel("MRR", fontsize=10)
ax_diff.set_title("MRR par difficulté", fontsize=12, fontweight="bold", color="#333")
ax_diff.set_facecolor(BG)
ax_diff.grid(axis="y", alpha=0.3)
ax_diff.spines[["top", "right"]].set_visible(False)
for bar, val in zip(bars2, diff_mrr):
    ax_diff.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=10, color=COLOR_D)

# ── Pie : succès / échec ──────────────────────────────────────────────────────
ok_count  = sum(1 for r in results if r["mrr"] > 0 or not r["expected_apis"])
nok_count = n - ok_count
ax_pie.pie(
    [ok_count, nok_count],
    labels=[f"Réussies\n({ok_count})", f"Ratées\n({nok_count})"],
    colors=[COLOR, "#F09595"],
    autopct="%1.1f%%", startangle=90,
    textprops={"fontsize": 10},
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
ax_pie.set_title("Répartition succès/échec", fontsize=12, fontweight="bold", color="#333")
ax_pie.set_facecolor(BG)

plt.suptitle(
    f"Évaluation RAG — Agentic4API | {AGENT_NAME} | {n} questions · 150 APIs",
    fontsize=14, fontweight="bold", color="#222", y=0.99
)
plt.savefig("rag_evaluation_gemini_pinecone.png", dpi=150,
            bbox_inches="tight", facecolor=BG)
plt.show()
print("✅ Dashboard sauvegardé → rag_evaluation_gemini_pinecone.png")

## 💾 Cellule 8 — Sauvegarde et téléchargement

Sauvegarde les résultats complets en JSON et télécharge :
- `evaluation_gemini_pinecone_150.json` — résultats bruts + métriques
- `rag_evaluation_gemini_pinecone.png` — dashboard

In [ ]:
import json

output = {
    "agent":          AGENT_NAME,
    "total_questions": n,
    "global_metrics": global_metrics,
    "by_category":    rows_cat,
    "by_difficulty":  rows_diff,
    "failed_questions": [
        {"id": r["id"], "question": r["question"],
         "expected": r["expected_apis"], "retrieved": r["retrieved_apis"]}
        for r in failed
    ],
    "raw_results": results
}

with open("evaluation_gemini_pinecone_150.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

# Téléchargement depuis Colab
try:
    from google.colab import files
    files.download("evaluation_gemini_pinecone_150.json")
    files.download("rag_evaluation_gemini_pinecone.png")
    print("✅ Fichiers téléchargés")
except ImportError:
    print("✅ Fichiers sauvegardés localement (pas dans Colab)")

## 🔴 Cellule 9 — Configuration Sans RAG

> **Avant de lancer** : Colle ici l'URL de ton workflow n8n sans Pinecone.

Dans n8n : **Dupliquer le workflow RAG → Supprimer le node Pinecone → Activer le webhook → Copier l'URL**

In [ ]:
# ⚠️ REMPLACE PAR L'URL DE TON WORKFLOW N8N SANS PINECONE
AGENT_URL_NO_RAG  = "https://nexdigital.app.n8n.cloud/webhook/TON-WEBHOOK-SANS-RAG/chat"
AGENT_NAME_NO_RAG = "Gemini Pro — Sans RAG (Prompt Stuffing)"

print(f"✅ Agent sans RAG configuré")
print(f"   URL : {AGENT_URL_NO_RAG}")

# Test rapide pour vérifier que le webhook répond
resp_test = call_agent(AGENT_URL_NO_RAG, "Quelle API pour créer une commande ?")
print(f"\n🔍 Test sans RAG :")
print(f"   Réponse : {resp_test['answer'][:200]}")
print(f"   Erreur  : {resp_test['error']}")

## 🚀 Cellule 10 — Évaluation Sans RAG (100 questions)

**⏱ Durée estimée** : ~3 min (même durée que le RAG)

> Gemini reçoit les 150 APIs directement dans le system prompt — pas de Pinecone.

In [ ]:
results_no_rag = []
TOTAL = len(GOLDEN_DATASET)

print(f"\n🚀 Évaluation {AGENT_NAME_NO_RAG} — {TOTAL} questions\n")

for i, item in enumerate(GOLDEN_DATASET):
    print(f"[{i+1:3}/{TOTAL}] {item['id']}  ({item['category']})...", end="  ")

    t0   = time.time()
    resp = call_agent(AGENT_URL_NO_RAG, item["question"])
    lat  = round(time.time() - t0, 2)

    retrieved = extract_apis(resp["answer"])
    p  = precision_at_k(retrieved, item["expected_apis"], TOP_K)
    r  = recall_at_k(retrieved,   item["expected_apis"], TOP_K)
    m  = mrr(retrieved,           item["expected_apis"])

    results_no_rag.append({
        **item,
        "retrieved_apis": retrieved,
        "answer":   resp["answer"],
        "contexts": resp.get("contexts", []),
        "precision": round(p, 3),
        "recall":    round(r, 3),
        "mrr":       round(m, 3),
        "latency":   lat,
        "error":     resp["error"],
    })

    status = "✅" if m > 0 or not item["expected_apis"] else "❌"
    print(f"{status}  P={p:.2f}  R={r:.2f}  MRR={m:.2f}  ({lat}s)")

    if i < TOTAL - 1:
        time.sleep(1.5)

print(f"\n✅ Évaluation sans RAG terminée — {TOTAL} questions traitées")

## 📊 Cellule 11 — Métriques Sans RAG

In [ ]:
import pandas as pd

n_nr = len(results_no_rag)

# ── Métriques globales sans RAG ──────────────────────────────────────────────
global_metrics_no_rag = {
    "Agent":            AGENT_NAME_NO_RAG,
    "Questions":        n_nr,
    "Precision@K":      round(sum(r["precision"] for r in results_no_rag) / n_nr, 3),
    "Recall@K":         round(sum(r["recall"]    for r in results_no_rag) / n_nr, 3),
    "MRR":              round(sum(r["mrr"]       for r in results_no_rag) / n_nr, 3),
    "Latence moy. (s)": round(sum(r["latency"]   for r in results_no_rag) / n_nr, 2),
    "Erreurs":          sum(1 for r in results_no_rag if r["error"]),
    "Réponses vides":   sum(1 for r in results_no_rag if not r["answer"]),
    "Questions réussies (MRR>0)": sum(1 for r in results_no_rag if r["mrr"] > 0 or not r["expected_apis"]),
}

print("\n" + "="*60)
print(f"  RÉSULTATS — {AGENT_NAME_NO_RAG}")
print("="*60)
for k, v in global_metrics_no_rag.items():
    print(f"  {k:<30} : {v}")

# ── Par catégorie sans RAG ────────────────────────────────────────────────────
cats_nr = sorted(set(r["category"] for r in results_no_rag))
rows_cat_nr = []
for cat in cats_nr:
    subset = [r for r in results_no_rag if r["category"] == cat]
    rows_cat_nr.append({
        "Catégorie":   cat,
        "N":           len(subset),
        "Precision@K": round(sum(r["precision"] for r in subset) / len(subset), 3),
        "Recall@K":    round(sum(r["recall"]    for r in subset) / len(subset), 3),
        "MRR":         round(sum(r["mrr"]       for r in subset) / len(subset), 3),
    })

df_cat_nr = pd.DataFrame(rows_cat_nr).set_index("Catégorie")
print("\n" + "="*60)
print("  PAR CATÉGORIE — Sans RAG")
print("="*60)
print(df_cat_nr.to_string())

# ── Par difficulté sans RAG ───────────────────────────────────────────────────
diffs_nr = sorted(set(r["difficulty"] for r in results_no_rag))
rows_diff_nr = []
for diff in diffs_nr:
    subset = [r for r in results_no_rag if r["difficulty"] == diff]
    rows_diff_nr.append({
        "Difficulté":  diff,
        "N":           len(subset),
        "MRR":         round(sum(r["mrr"]       for r in subset) / len(subset), 3),
        "Recall@K":    round(sum(r["recall"]    for r in subset) / len(subset), 3),
        "Precision@K": round(sum(r["precision"] for r in subset) / len(subset), 3),
    })

df_diff_nr = pd.DataFrame(rows_diff_nr).set_index("Difficulté")
print("\n" + "="*60)
print("  PAR DIFFICULTÉ — Sans RAG")
print("="*60)
print(df_diff_nr.to_string())

## ⚖️ Cellule 12 — Tableau Comparatif RAG vs Sans RAG

**C'est le tableau central du mémoire** — il prouve que le RAG apporte de la valeur.

In [ ]:
import pandas as pd

# ── Tableau global ────────────────────────────────────────────────────────────
df_global_comp = pd.DataFrame([
    {
        "Agent":             "✅ Avec RAG (Gemini + Pinecone)",
        "MRR":               global_metrics["MRR"],
        "Recall@K":          global_metrics["Recall@K"],
        "Precision@K":       global_metrics["Precision@K"],
        "Latence moy. (s)":  global_metrics["Latence moy. (s)"],
        "Questions réussies": f"{global_metrics['Questions réussies (MRR>0)']}/100",
        "Erreurs":           global_metrics["Erreurs"],
    },
    {
        "Agent":             "🔴 Sans RAG (Prompt Stuffing)",
        "MRR":               global_metrics_no_rag["MRR"],
        "Recall@K":          global_metrics_no_rag["Recall@K"],
        "Precision@K":       global_metrics_no_rag["Precision@K"],
        "Latence moy. (s)":  global_metrics_no_rag["Latence moy. (s)"],
        "Questions réussies": f"{global_metrics_no_rag['Questions réussies (MRR>0)']}/100",
        "Erreurs":           global_metrics_no_rag["Erreurs"],
    },
]).set_index("Agent")

print("\n" + "="*70)
print("  COMPARAISON GLOBALE — RAG vs SANS RAG")
print("="*70)
print(df_global_comp.to_string())

# ── Delta RAG - Sans RAG ─────────────────────────────────────────────────────
delta_mrr   = round(global_metrics["MRR"]         - global_metrics_no_rag["MRR"],         3)
delta_rec   = round(global_metrics["Recall@K"]    - global_metrics_no_rag["Recall@K"],    3)
delta_prec  = round(global_metrics["Precision@K"] - global_metrics_no_rag["Precision@K"], 3)

print(f"\n  Gains du RAG sur le Sans RAG :")
print(f"  MRR          : {'+' if delta_mrr>=0 else ''}{delta_mrr:+.3f}  ({'RAG gagne' if delta_mrr>0 else 'Sans RAG gagne'})")
print(f"  Recall@K     : {'+' if delta_rec>=0 else ''}{delta_rec:+.3f}  ({'RAG gagne' if delta_rec>0 else 'Sans RAG gagne'})")
print(f"  Precision@K  : {'+' if delta_prec>=0 else ''}{delta_prec:+.3f}  ({'RAG gagne' if delta_prec>0 else 'Sans RAG gagne'})")

# ── Tableau par catégorie ─────────────────────────────────────────────────────
cats_all = sorted(set(r["category"] for r in results))
rows_comp_cat = []
for cat in cats_all:
    rag_sub = [r for r in results        if r["category"] == cat]
    nr_sub  = [r for r in results_no_rag if r["category"] == cat]
    mrr_rag = round(sum(r["mrr"] for r in rag_sub) / len(rag_sub), 3)
    mrr_nr  = round(sum(r["mrr"] for r in nr_sub)  / len(nr_sub),  3)
    delta   = round(mrr_rag - mrr_nr, 3)
    rows_comp_cat.append({
        "Catégorie":    cat,
        "N":            len(rag_sub),
        "MRR RAG":      mrr_rag,
        "MRR Sans RAG": mrr_nr,
        "Delta":        f"{'+' if delta>=0 else ''}{delta:.3f}",
        "Vainqueur":    "RAG ✅" if delta > 0 else ("Égalité" if delta == 0 else "Sans RAG"),
    })

df_comp_cat = pd.DataFrame(rows_comp_cat).set_index("Catégorie")
print("\n" + "="*70)
print("  PAR CATÉGORIE — RAG vs SANS RAG")
print("="*70)
print(df_comp_cat.to_string())

# ── Tableau par difficulté ────────────────────────────────────────────────────
diffs_all = sorted(set(r["difficulty"] for r in results))
rows_comp_diff = []
for diff in diffs_all:
    rag_sub = [r for r in results        if r["difficulty"] == diff]
    nr_sub  = [r for r in results_no_rag if r["difficulty"] == diff]
    mrr_rag = round(sum(r["mrr"] for r in rag_sub) / len(rag_sub), 3)
    mrr_nr  = round(sum(r["mrr"] for r in nr_sub)  / len(nr_sub),  3)
    rows_comp_diff.append({
        "Difficulté":   diff,
        "N":            len(rag_sub),
        "MRR RAG":      mrr_rag,
        "MRR Sans RAG": mrr_nr,
        "Delta RAG":    f"{'+' if mrr_rag-mrr_nr>=0 else ''}{mrr_rag-mrr_nr:.3f}",
    })

df_comp_diff = pd.DataFrame(rows_comp_diff).set_index("Difficulté")
print("\n" + "="*70)
print("  PAR DIFFICULTÉ — RAG vs SANS RAG")
print("="*70)
print(df_comp_diff.to_string())

## 📈 Cellule 13 — Dashboard Comparatif RAG vs Sans RAG

Dashboard matplotlib côte-à-côte pour le mémoire.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

BG      = "#f8f9fa"
C_RAG   = "#378ADD"   # bleu  — avec RAG
C_NORAG = "#E24B4A"   # rouge — sans RAG
C_RAG_D = "#185FA5"
C_NR_D  = "#A32D2D"

fig = plt.figure(figsize=(20, 16))
fig.patch.set_facecolor(BG)
gs  = fig.add_gridspec(3, 4, hspace=0.55, wspace=0.4)

ax_kpi  = fig.add_subplot(gs[0, :])    # KPI cards côte-à-côte
ax_cat  = fig.add_subplot(gs[1, :])    # MRR par catégorie groupé
ax_diff = fig.add_subplot(gs[2, :2])   # MRR par difficulté groupé
ax_tok  = fig.add_subplot(gs[2, 2])    # Tokens / coût
ax_pie  = fig.add_subplot(gs[2, 3])    # Delta MRR catégories

# ── KPI cards côte à côte ─────────────────────────────────────────────────────
ax_kpi.set_facecolor(BG)
ax_kpi.axis("off")

metrics_pairs = [
    ("MRR",          global_metrics["MRR"],          global_metrics_no_rag["MRR"]),
    ("Recall@K",     global_metrics["Recall@K"],     global_metrics_no_rag["Recall@K"]),
    ("Precision@K",  global_metrics["Precision@K"],  global_metrics_no_rag["Precision@K"]),
    ("Latence (s)",  global_metrics["Latence moy. (s)"], global_metrics_no_rag["Latence moy. (s)"]),
    ("Réussies",     global_metrics["Questions réussies (MRR>0)"], global_metrics_no_rag["Questions réussies (MRR>0)"]),
]

xs = np.linspace(0.08, 0.92, len(metrics_pairs))
for x, (label, v_rag, v_nr) in zip(xs, metrics_pairs):
    # Fond
    for offset, color, val, name in [(-0.045, C_RAG, v_rag, "RAG"), (+0.045, C_NORAG, v_nr, "No RAG")]:
        bg_col = "#E6F1FB" if color == C_RAG else "#FCEBEB"
        box = mpatches.FancyBboxPatch(
            (x + offset - 0.038, 0.08), 0.076, 0.84,
            boxstyle="round,pad=0.01", transform=ax_kpi.transAxes,
            facecolor=bg_col, edgecolor=color, linewidth=0.8
        )
        ax_kpi.add_patch(box)
        ax_kpi.text(x + offset, 0.80, name, ha="center", va="center",
                    fontsize=9, color=color, transform=ax_kpi.transAxes)
        fmt = f"{val:.3f}" if label != "Réussies" else f"{int(val)}/100"
        ax_kpi.text(x + offset, 0.46, fmt, ha="center", va="center",
                    fontsize=18, color=color, fontweight="bold", transform=ax_kpi.transAxes)
    ax_kpi.text(x, 0.15, label, ha="center", va="center",
                fontsize=10, color="#555", transform=ax_kpi.transAxes)
ax_kpi.set_title("Comparaison globale — RAG vs Sans RAG · 100 questions · 150 APIs",
                  fontsize=13, fontweight="bold", color="#222", pad=10)

# ── MRR par catégorie groupé ──────────────────────────────────────────────────
cats_labels = [r["Catégorie"] for r in rows_comp_cat]
mrr_rag_cat = [r["MRR RAG"]      for r in rows_comp_cat]
mrr_nr_cat  = [r["MRR Sans RAG"] for r in rows_comp_cat]

x  = np.arange(len(cats_labels))
w  = 0.38
b1 = ax_cat.bar(x - w/2, mrr_rag_cat, w, color=C_RAG,   alpha=0.88, label="Avec RAG")
b2 = ax_cat.bar(x + w/2, mrr_nr_cat,  w, color=C_NORAG, alpha=0.88, label="Sans RAG")
ax_cat.set_xticks(x)
ax_cat.set_xticklabels(cats_labels, fontsize=9, rotation=22, ha="right")
ax_cat.set_ylim(0, 1.18)
ax_cat.set_ylabel("MRR", fontsize=10)
ax_cat.set_title("MRR par catégorie — RAG vs Sans RAG", fontsize=12, fontweight="bold", color="#333")
ax_cat.legend(fontsize=9)
ax_cat.set_facecolor(BG)
ax_cat.grid(axis="y", alpha=0.3)
ax_cat.spines[["top", "right"]].set_visible(False)
for bar, val in zip(b1, mrr_rag_cat):
    ax_cat.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f"{val:.2f}", ha="center", va="bottom", fontsize=7.5, color=C_RAG_D)
for bar, val in zip(b2, mrr_nr_cat):
    ax_cat.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f"{val:.2f}", ha="center", va="bottom", fontsize=7.5, color=C_NR_D)

# ── MRR par difficulté ────────────────────────────────────────────────────────
diff_labels  = [r["Difficulté"]  for r in rows_comp_diff]
mrr_rag_diff = [r["MRR RAG"]      for r in rows_comp_diff]
mrr_nr_diff  = [r["MRR Sans RAG"] for r in rows_comp_diff]

x2 = np.arange(len(diff_labels))
b3 = ax_diff.bar(x2 - w/2, mrr_rag_diff, w, color=C_RAG,   alpha=0.88, label="Avec RAG")
b4 = ax_diff.bar(x2 + w/2, mrr_nr_diff,  w, color=C_NORAG, alpha=0.88, label="Sans RAG")
ax_diff.set_xticks(x2)
ax_diff.set_xticklabels(diff_labels, fontsize=11)
ax_diff.set_ylim(0, 1.15)
ax_diff.set_ylabel("MRR", fontsize=10)
ax_diff.set_title("MRR par difficulté", fontsize=12, fontweight="bold", color="#333")
ax_diff.legend(fontsize=9)
ax_diff.set_facecolor(BG)
ax_diff.grid(axis="y", alpha=0.3)
ax_diff.spines[["top", "right"]].set_visible(False)
for bar, val in zip(b3, mrr_rag_diff):
    ax_diff.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=9, color=C_RAG_D)
for bar, val in zip(b4, mrr_nr_diff):
    ax_diff.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=9, color=C_NR_D)

# ── Tokens / question ─────────────────────────────────────────────────────────
tok_labels = ["Avec RAG", "Sans RAG"]
tok_values = [2000, 60000]
tok_colors = [C_RAG, C_NORAG]
bars_tok = ax_tok.bar(tok_labels, tok_values, color=tok_colors, alpha=0.85, width=0.5)
ax_tok.set_ylabel("Tokens / question", fontsize=10)
ax_tok.set_title("Coût tokens", fontsize=12, fontweight="bold", color="#333")
ax_tok.set_facecolor(BG)
ax_tok.grid(axis="y", alpha=0.3)
ax_tok.spines[["top", "right"]].set_visible(False)
for bar, val in zip(bars_tok, tok_values):
    ax_tok.text(bar.get_x() + bar.get_width()/2, val + 500,
                f"~{val:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax_tok.set_ylim(0, 75000)

# ── Delta MRR par catégorie (horizontal) ─────────────────────────────────────
delta_vals  = [float(r["Delta"].replace("+","")) for r in rows_comp_cat]
delta_cols  = [C_RAG if d >= 0 else C_NORAG for d in delta_vals]
y_pos = np.arange(len(cats_labels))
ax_pie.barh(y_pos, delta_vals, color=delta_cols, alpha=0.85)
ax_pie.set_yticks(y_pos)
ax_pie.set_yticklabels(cats_labels, fontsize=8)
ax_pie.axvline(0, color="#888", linewidth=0.8, linestyle="--")
ax_pie.set_xlabel("Delta MRR (RAG − Sans RAG)", fontsize=9)
ax_pie.set_title("Avantage RAG par catégorie", fontsize=12, fontweight="bold", color="#333")
ax_pie.set_facecolor(BG)
ax_pie.spines[["top", "right"]].set_visible(False)
for i, (val, col) in enumerate(zip(delta_vals, delta_cols)):
    ax_pie.text(val + (0.005 if val >= 0 else -0.005), i,
                f"{'+' if val >= 0 else ''}{val:.3f}",
                va="center", ha="left" if val >= 0 else "right",
                fontsize=7.5, color=col, fontweight="bold")

plt.suptitle(
    "Agentic4API — RAG vs Sans RAG | Gemini Pro + Pinecone | 100 questions · 150 APIs",
    fontsize=13, fontweight="bold", color="#222", y=0.99
)
plt.savefig("rag_vs_no_rag_dashboard.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("✅ Dashboard comparatif sauvegardé → rag_vs_no_rag_dashboard.png")

## 💾 Cellule 14 — Sauvegarde Complète (RAG + Sans RAG)

Sauvegarde tous les résultats dans un seul fichier JSON.

In [ ]:
import json

output_final = {
    "with_rag": {
        "agent":           AGENT_NAME,
        "total_questions":  len(results),
        "global_metrics":   global_metrics,
        "by_category":      rows_cat,
        "by_difficulty":    rows_diff,
        "raw_results":      results,
    },
    "without_rag": {
        "agent":           AGENT_NAME_NO_RAG,
        "total_questions":  len(results_no_rag),
        "global_metrics":   global_metrics_no_rag,
        "by_category":      rows_cat_nr,
        "by_difficulty":    rows_diff_nr,
        "raw_results":      results_no_rag,
    },
    "comparison": {
        "delta_mrr":        round(global_metrics["MRR"] - global_metrics_no_rag["MRR"], 3),
        "delta_recall":     round(global_metrics["Recall@K"] - global_metrics_no_rag["Recall@K"], 3),
        "delta_precision":  round(global_metrics["Precision@K"] - global_metrics_no_rag["Precision@K"], 3),
        "by_category":      rows_comp_cat,
        "by_difficulty":    rows_comp_diff,
        "tokens_with_rag":    2000,
        "tokens_without_rag": 60000,
        "token_reduction_pct": round((1 - 2000/60000) * 100, 1),
    }
}

with open("agentic4api_full_evaluation.json", "w", encoding="utf-8") as f:
    json.dump(output_final, f, ensure_ascii=False, indent=2)

print("✅ Résultats complets sauvegardés → agentic4api_full_evaluation.json")
print(f"\n  RÉSUMÉ FINAL :")
print(f"  RAG     — MRR: {global_metrics['MRR']} | Recall: {global_metrics['Recall@K']} | Precision: {global_metrics['Precision@K']}")
print(f"  Sans RAG — MRR: {global_metrics_no_rag['MRR']} | Recall: {global_metrics_no_rag['Recall@K']} | Precision: {global_metrics_no_rag['Precision@K']}")
print(f"  Delta MRR     : {round(global_metrics['MRR'] - global_metrics_no_rag['MRR'], 3):+.3f}")
print(f"  Réduction tokens : -96.7% ({60000:,} → {2000:,} tokens/question)")

try:
    from google.colab import files
    files.download("agentic4api_full_evaluation.json")
    files.download("rag_vs_no_rag_dashboard.png")
    print("\n✅ Fichiers téléchargés")
except ImportError:
    print("\n✅ Fichiers sauvegardés localement")

## 🔬 Cellule 15 — Installation ColBERT (RAGatouille)

> **GPU T4 requis** — Vérifie que ton runtime est bien en GPU : Runtime → Changer le type d'exécution → T4 GPU

RAGatouille encapsule ColBERT v2.0 (Khattab & Zaharia 2020, Stanford NLP).
Le modèle `colbert-ir/colbertv2.0` sera téléchargé automatiquement (~440 MB).

In [ ]:
# ── Vérifier le GPU ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                        '--format=csv,noheader'], capture_output=True, text=True)
print(f'GPU détecté : {result.stdout.strip()}')

# ── Désinstaller tout ce qui conflicte ───────────────────────────────────────
!pip uninstall -q -y ragatouille langchain langchain-community langchain-core langchain-text-splitters 2>/dev/null

# ── Installer dans le bon ordre avec versions exactes ────────────────────────
!pip install -q "langchain-core==0.1.52"
!pip install -q "langchain==0.1.20"
!pip install -q "langchain-community==0.0.38"
!pip install -q "transformers==4.36.2"
!pip install -q "ragatouille==0.0.8.post2"

print('\n✅ Toutes les dépendances installées')
print('⚠️  Maintenant : Runtime → Restart session → puis lance la cellule suivante')

GPU détecté : Tesla T4, 15360 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.9/302.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigquery-magics 0.14.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
wheel 0.46.3 requires packaging>=24.0, but you have packaging 23.2 which is incompatible.
langgraph 1.1.8 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.1.52 which is incompatible.
langgraph-checkpoint 4.0.2 requires langchain-core>=0.2.38, but you have langchain-core 0.1.52 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.1.52 which is incompatible.
google-adk 1.29.0

In [ ]:
# ── Vérification post-restart (cellule séparée) ──────────────────────────────
import warnings
warnings.filterwarnings('ignore')

from ragatouille import RAGPretrainedModel
import torch

print(f'✅ RAGatouille importé')
print(f'✅ PyTorch  : {torch.__version__}')
print(f'✅ CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU      : {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

✅ RAGatouille importé
✅ PyTorch  : 2.10.0+cu128
✅ CUDA     : True
✅ GPU      : Tesla T4
✅ VRAM     : 15.6 GB


## 📚 Cellule 16 — Construction du corpus ColBERT

ColBERT indexe des **documents textuels**. On construit un document par API
en combinant toutes les informations disponibles dans nos fichiers JSON.

In [ ]:
# ── Upload et extraction du ZIP ──────────────────────────────────────────────
from google.colab import files
import zipfile, os

print("📁 Sélectionne le fichier api-catalogue-150-FINAL.zip")
uploaded = files.upload()  # ← fenêtre qui s'ouvre

# Extraction
zip_name = list(uploaded.keys())[0]
print(f'\n📦 Extraction de {zip_name}...')

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# Trouver le bon dossier extrait
extracted = [d for d in os.listdir('/content/') if 'api' in d.lower() and os.path.isdir(f'/content/{d}')]
print(f'Dossiers trouvés : {extracted}')

# Renommer en apis150
for d in extracted:
    if d != 'apis150':
        os.rename(f'/content/{d}', '/content/apis150')
        print(f'✅ Renommé {d} → apis150')

count = len([f for f in os.listdir('/content/apis150') if f.endswith('.json')])
print(f'✅ {count} fichiers JSON prêts dans /content/apis150/')

📁 Sélectionne le fichier api-catalogue-150-FINAL.zip


Saving api-catalogue-150 (2).zip to api-catalogue-150 (2).zip

📦 Extraction de api-catalogue-150 (2).zip...
Dossiers trouvés : ['api-catalogue']
✅ Renommé api-catalogue → apis150
✅ 150 fichiers JSON prêts dans /content/apis150/


In [ ]:
import json, os, re
from collections import Counter

API_DIR = '/content/apis150'   # ← chemin vers tes JSON uploadés

# ── Fonction pour construire le texte d'un document ColBERT ─────────────────
def build_colbert_doc(api: dict, filename: str) -> dict:
    info    = api.get('info', {})
    api_id  = info.get('x-api-id', filename.replace('.json', ''))
    title   = info.get('title', api_id)
    desc    = info.get('description', '')
    team    = info.get('x-team', '')
    domain  = info.get('x-domain', '')
    version = info.get('version', '')
    status  = info.get('x-status', 'active')
    fp      = info.get('x-false-positive-warning', '')
    bf      = info.get('x-breaking-changes-from-previous', '')
    succ    = info.get('x-successor', '')

    # Endpoints avec leurs summaries
    paths = api.get('paths', {})
    endpoint_lines = []
    for path, methods in paths.items():
        for method, op in methods.items():
            if method in ('get','post','put','patch','delete') and isinstance(op, dict):
                summary = op.get('summary', '')
                endpoint_lines.append(f'{method.upper()} {path}: {summary}')

    # Construction du texte
    parts = [f'{title} (version {version}) statut {status}.', desc]
    if fp:   parts.append(f'FAUX POSITIFS: {fp}')
    if bf:
        bc = bf if isinstance(bf, str) else '. '.join(bf)
        parts.append(f'BREAKING CHANGES: {bc}')
    if succ: parts.append(f'Successeur: {succ}')
    parts.append(f'Equipe: {team}. Domaine: {domain}.')
    if endpoint_lines:
        parts.append('ENDPOINTS: ' + ' | '.join(endpoint_lines[:15]))

    text = ' '.join(p for p in parts if p).strip()
    return {'api_id': api_id, 'text': text}


# ── Charger les JSONs ────────────────────────────────────────────────────────
corpus_docs_raw = []   # (api_id, text)
load_errors     = []

if os.path.exists(API_DIR):
    files_json = sorted([f for f in os.listdir(API_DIR) if f.endswith('.json')])
    print(f'📁 Dossier trouvé : {API_DIR} ({len(files_json)} fichiers JSON)')
    for filename in files_json:
        try:
            with open(os.path.join(API_DIR, filename), encoding='utf-8') as f:
                api = json.load(f)
            doc = build_colbert_doc(api, filename)
            corpus_docs_raw.append((doc['api_id'], doc['text']))
        except Exception as e:
            load_errors.append((filename, str(e)))
    if load_errors:
        print(f'⚠️  {len(load_errors)} erreurs de chargement :')
        for fn, err in load_errors[:5]:
            print(f'   {fn}: {err}')
else:
    print(f'⚠️  Dossier {API_DIR} non trouvé — utilisation du catalogue compact')
    CATALOGUE_COMPACT = {
        'order-api-v4': 'Order API v4 active. Commandes e-commerce multi-articles. POST /v4/orders créer. PATCH /v4/orders/{id}/status mettre à jour statut. PUT /v4/orders/{id}/cancel annuler. POST /v4/orders/{id}/refund rembourser.',
        'order-api-v3': 'Order API v3 deprecated migrer vers v4. Commandes avec adresse structurée discount.',
        'order-api-v2': 'Order API v2 deprecated migrer vers v4. Commandes multi-articles basiques.',
        'order-api-v1': 'Order API v1 deprecated migrer vers v4. Commandes mono-produit.',
        'cart-api': 'Cart API active. Panier achat temporaire. POST /v1/cart/{userId}/items ajouter. POST /v1/cart/{userId}/checkout valider créer commande. DIFFÉRENCE vs order-api: Cart temporaire avant achat.',
        'checkout-api': 'Checkout API active. Tunnel achat étapes. POST /v1/checkout/start. PUT adresse. GET shipping. POST confirmer commande.',
        'product-api-v4': 'Product API v4 active. Produits multi-catalogue B2B B2C IA description. POST /v4/products. POST /v4/products/{id}/ai-description générer description IA. lifecycle stage.',
        'product-api-v3': 'Product API v3 deprecated. Produits SEO variantes sous-ressource.',
        'product-api-v2': 'Product API v2 deprecated. Produits variantes intégrées.',
        'product-api-v1': 'Product API v1 deprecated.',
        'product-catalog-api': 'Product Catalog API active. Taxonomie catégories produits hiérarchie. GET /v2/categories. DIFFÉRENCE vs catalog-api: Product Catalog = taxonomie structurée.',
        'catalog-api': 'Catalog API active. Collections publiées B2B B2C canal. POST /v1/catalogs. POST /v1/catalogs/{id}/publish. DIFFÉRENCE vs product-catalog-api: Catalog = collections par canal.',
        'pricing-api': 'Pricing API active. Prix dynamiques remises promotions. GET /v1/pricing/product/{id}. POST /v1/pricing/calculate. POST /v1/pricing/promo/validate code promo.',
        'price-list-api': 'Price List API active. Listes prix contractuels B2B. POST /v1/price-lists lookup.',
        'discount-api': 'Discount API active. Codes réduction. POST /v1/discounts créer code. GET /v1/discounts/{code} valider.',
        'promotion-api': 'Promotion API active. Promotions automatiques sans code. POST /v1/promotions/eligible.',
        'search-api-v3': 'Search API v3 active. Recherche hybride LLM conversationnel synonymes merchandising. GET /v3/search mode conversational lexical semantic. DIFFÉRENCE vs search-api-v2: v3 ajoute LLM.',
        'search-api-v2': 'Search API v2 deprecated. Recherche sémantique vectorielle.',
        'search-api': 'Search API deprecated. Recherche full-text basique.',
        'review-api': 'Review API active. Avis textuels produits clients. POST /v1/reviews. GET /v1/reviews/product/{id}. PUT /v1/reviews/{id}/moderate modérer.',
        'rating-api': 'Rating API active. Notes numériques produits. POST /v1/ratings. GET /v1/ratings/average/{productId}.',
        'bundle-api': 'Bundle API active. Packs produits groupés. POST /v1/bundles. GET /v1/bundles/{id}/price.',
        'gift-card-api': 'Gift Card API active. Cartes cadeaux. POST /v1/gift-cards/issue. POST /v1/gift-cards/{code}/redeem.',
        'pre-order-api': 'Pre-Order API active. Pré-commandes avant disponibilité. POST /v1/pre-orders.',
        'recommendation-api': 'Recommendation API active. Recommandations ML personnalisées. GET /v1/recommendations/{userId}. GET /v1/recommendations/trending.',
        'stock-reservation-api': 'Stock Reservation API active. Réservation temporaire stock. POST /v1/stock/reserve. POST /v1/stock/confirm.',
        'shipping-api-v2': 'Shipping API v2 active. Expéditions multi-colis GPS temps réel CO2. POST /v2/shipping/create. GET /v2/shipping/{id}/live-track suivi GPS. ZPL PDF étiquettes. DIFFÉRENCE vs delivery-api: Shipping = colis transporteurs.',
        'shipping-api': 'Shipping API deprecated migrer vers v2. Expéditions mono-colis.',
        'delivery-api': 'Delivery API active. Créneaux livraison planifiés. GET /v1/delivery/slots disponibilités. POST book réserver. DIFFÉRENCE vs shipping-api: Delivery = planning créneaux.',
        'return-api': 'Return API active. Retours produits clients. POST /v1/returns. PUT /v1/returns/{id}/approve approuver retour remboursement.',
        'inventory-api': 'Inventory API v3 active. Stocks temps réel alertes. GET /v3/inventory/{productId}. PATCH /v3/inventory/{productId}/stock mettre à jour. POST /v3/inventory/restock réapprovisionner.',
        'inventory-api-v2': 'Inventory API v2 deprecated. Stocks historique mouvements alertes seuil.',
        'inventory-api-v1': 'Inventory API v1 deprecated. Stocks basiques.',
        'warehouse-api': 'Warehouse API active. Entrepôts emplacements physiques. POST /v1/warehouses/transfer transfert entre entrepôts.',
        'logistics-tracking-api': 'Logistics Tracking API active. Tracking unifié multi-transporteurs. GET /v1/tracking/{code}. POST /v1/tracking/batch.',
        'store-locator-api': 'Store Locator API active. Points vente géolocalisation stock. GET /v1/stores/nearby. GET /v1/stores/{id}/stock/{productId}.',
        'supplier-api': 'Supplier API active. Fournisseurs B2B évaluation. POST /v1/suppliers. POST /v1/suppliers/{id}/evaluate.',
        'purchase-order-api': 'Purchase Order API active. Bons commande fournisseurs achats. POST /v1/purchase-orders créer bon. POST /v1/purchase-orders/{id}/receive réception stock. DIFFÉRENCE vs order-api: Purchase Order = commandes vers fournisseurs.',
        'demand-forecast-api': 'Demand Forecast API active. Prévisions ML demande stock. POST /v1/forecast/demand prévisions jours. GET /v1/forecast/stockouts risque rupture.',
        'quality-api': 'Quality API active. Contrôle qualité réceptions. POST /v1/quality/inspections. POST /v1/quality/non-conformities.',
        'payment-api-v3': 'Payment API v3 active. Paiements card PayPal SEPA BNPL bitcoin ethereum USDC disputes réconciliation. POST /v3/payments. POST /v3/payments/{id}/refund. POST /v3/payments/{id}/dispute chargeback. DIFFÉRENCE vs billing: Payment = transaction unique.',
        'payment-api-v2': 'Payment API v2 deprecated migrer vers v3. Paiements card PayPal SEPA capture manuel remboursement partiel.',
        'payment-api-v1': 'Payment API v1 deprecated. Paiements carte uniquement.',
        'payment-api': 'Payment API deprecated. Alias payment-api-v2.',
        'billing-api-v3': 'Billing API v3 active. Abonnements récurrents flat metered hybrid. POST /v3/billing/subscriptions prélèvement automatique mensuel. POST /v3/billing/usage consommation. POST /v3/billing/erp-sync synchroniser SAP Oracle Sage CEGID ERP. POST /v3/billing/credits avoir. dunning relance impayé. DIFFÉRENCE vs payment: Billing = abonnements récurrents.',
        'billing-api-v2': 'Billing API v2 deprecated. Billing portail self-service.',
        'billing-api': 'Billing API deprecated migrer vers v3.',
        'invoice-api-v3': 'Invoice API v3 active. Factures documents fiscaux PDF légaux. POST /v3/invoices générer. GET /v3/invoices/{id}/accounting-export exporter FEC DATEV SAGE comptabilité. POST /v3/invoices/{id}/penalty pénalité retard. signature eIDAS B2B. DIFFÉRENCE vs billing: Invoice = documents fiscaux.',
        'invoice-api-v2': 'Invoice API v2 deprecated. Factures multilingues avoirs.',
        'invoice-api': 'Invoice API deprecated migrer vers v3.',
        'wallet-api': 'Wallet API active. Portefeuille électronique interne. GET /v1/wallets/{userId}. POST topup créditer. POST debit débiter.',
        'tax-api': 'Tax API active. Calcul TVA conformité fiscale. POST /v1/tax/calculate. POST /v1/tax/validate-vat numéro TVA.',
        'escrow-api': 'Escrow API active. Paiements séquestre marketplace. POST /v1/escrow bloquer fonds. POST /v1/escrow/{id}/release libérer.',
        'payout-api': 'Payout API active. Virements vers vendeurs partenaires SEPA. POST /v1/payouts. POST /v1/payouts/batch masse. DIFFÉRENCE vs payment: Payout = sortant vers tiers.',
        'subscription-api': 'Subscription API active. Plans droits accès. POST /v2/subscriptions. PUT upgrade downgrade.',
        'user-api-v3': 'User API v3 active. Utilisateurs credentials login SCIM 2.0 groupes SSO IdP provisioning. POST/GET /v3/users. GET/POST /v3/groups. GET/POST /v3/scim/v2/Users. DIFFÉRENCE vs customer-profile: User = identité technique credentials.',
        'user-api': 'User API v2 deprecated migrer vers v3. Utilisateurs avec 2FA.',
        'user-api-v1': 'User API v1 deprecated. Utilisateurs basiques.',
        'auth-api': 'Auth API v2 active. Authentification JWT OAuth2 refresh token. POST /v2/auth/login connexion. POST /v2/auth/refresh renouveler. scopes Bearer token header Authorization.',
        'auth-api-v1': 'Auth API v1 deprecated. Sessions serveur cookies.',
        'sso-api': 'SSO API active. Fédération SSO SAML2 OIDC Azure AD Okta Google. POST /v1/sso/providers configurer. GET /v1/sso/login/{providerId} initier connexion SSO.',
        'mfa-api': 'MFA API active. Multi-facteurs TOTP SMS FIDO2 WebAuthn. POST /v1/mfa/enroll enrôler. POST /v1/mfa/verify vérifier OTP. scopes OAuth2 MFA.',
        'session-api': 'Session API active. Sessions actives révocation suspectes. GET /v1/sessions/{userId}. DELETE /v1/sessions/{userId}/revoke-all invalider toutes sessions. GET /v1/sessions/suspicious.',
        'account-api': 'Account API active. Organisations B2B membres facturation. POST /v1/accounts créer organisation. POST /v1/accounts/{id}/members inviter.',
        'profile-api': 'Profile API active. Profil public préférences UX avatar. GET/PUT /v1/profile/{userId}. PUT avatar. GET/PUT préférences thème langue timezone. DIFFÉRENCE vs user-api: Profile = affichage préférences.',
        'permission-api': 'Permission API active. RBAC contrôle accès rôles permissions. POST /v1/permissions. GET/POST /v1/roles. POST /v1/check vérifier autorisation resource action. POST /v1/users/{userId}/roles assigner rôle.',
        'encryption-api': 'Encryption API active. Chiffrement AES RSA vault rotation. POST /v1/encrypt. POST /v1/decrypt. POST /v1/keys/{id}/rotate rotation clé.',
        'customer-profile-api-v2': 'Customer Profile API v2 active. Profils commerciaux clients score propension consentement RGPD. GET/POST /v2/customers. GET /v2/customers/{id}/propensity. segmentation VIP. DIFFÉRENCE vs user-api: Customer Profile = données commerciales.',
        'customer-profile-api': 'Customer Profile API deprecated migrer vers v2. Profils clients segmentation fidélité.',
        'crm-contact-api': 'CRM Contact API active. Contacts 360 interactions historique commercial. GET/POST /v1/crm/contacts. POST /v1/crm/contacts/{id}/interactions.',
        'lead-api': 'Lead API active. Prospects pipeline scoring. GET/POST /v1/leads. PUT /v1/leads/{id}/qualify. GET /v1/leads/score/{id}.',
        'campaign-api': 'Campaign API active. Campagnes marketing multicanal. POST /v1/campaigns. PUT /v1/campaigns/{id}/launch. GET stats résultats.',
        'loyalty-points-api': 'Loyalty Points API active. Points fidélité earn redeem. GET /v1/loyalty/{customerId} solde. POST /v1/loyalty/earn attribuer. POST /v1/loyalty/redeem utiliser.',
        'segmentation-api': 'Segmentation API active. Segments dynamiques règles comportementales. POST /v1/segments. GET /v1/segments/{id}/members. DIFFÉRENCE vs tag-api: Segmentation = groupes dynamiques automatiques.',
        'referral-api': 'Referral API active. Parrainage codes récompenses. POST /v1/referrals/generate code. POST /v1/referrals/validate valider parrainage inscription.',
        'notification-api-v3': 'Notification API v3 active. Email SMS Push WhatsApp in-app triggers IA personnalisation. POST /v3/notifications/send. POST /v3/notifications/triggers déclenchement événementiel. DIFFÉRENCE vs messaging: Notification = unidirectionnel système vers user.',
        'notification-api-v2': 'Notification API v2 deprecated. Templates multilingues.',
        'notification-api': 'Notification API deprecated migrer vers v3.',
        'email-api': 'Email API active. Canal email transactionnel templates tracking. POST /v2/emails/send. POST /v2/emails/batch envoi masse.',
        'sms-api': 'SMS API active. Canal SMS OTP vérification. POST /v1/sms/send. POST /v1/sms/otp/send. POST /v1/sms/otp/verify.',
        'push-api': 'Push API active. Notifications push mobiles iOS Android. POST /v1/push/send. POST /v1/push/register-device.',
        'messaging-api': 'Messaging API active. Chat BIDIRECTIONNEL user↔user temps réel. POST /v1/messages. GET /v1/messages/conversations/{userId}. DIFFÉRENCE vs notification: Messaging = bidirectionnel entre utilisateurs.',
        'alert-api': 'Alert API active. Alertes OPS INTERNES équipes techniques PagerDuty Slack. POST /v1/alerts. PUT /v1/alerts/{id}/acknowledge. DIFFÉRENCE vs notification: Alert = interne équipes ops.',
        'webhook-api': 'Webhook API active. Webhooks événements sortants. POST /v1/webhooks. POST /v1/webhooks/{id}/test. GET /v1/webhooks/{id}/deliveries.',
        'live-chat-api': 'Live Chat API active. Chat temps réel client↔agent support. POST /v1/chat/sessions. POST /v1/chat/sessions/{id}/transfer escalade. PUT close.',
        'chatbot-api': 'Chatbot API active. Bot IA conversationnel escalade humain. POST /v1/chatbot/message. POST /v1/chatbot/escalate vers agent humain.',
        'contact-form-api': 'Contact Form API active. Formulaires contact web ticket auto. POST /v1/contact/submit.',
        'analytics-api-v3': 'Analytics API v3 active. Métriques business ML prédictif alertes export Looker Tableau PowerBI. POST /v3/analytics/predict prévisions. POST /v3/analytics/alerts seuils. détection anomalies. DIFFÉRENCE vs reporting: Analytics = temps réel exploration.',
        'analytics-api-v2': 'Analytics API v2 deprecated. Analytics streaming SSE dashboards.',
        'analytics-api': 'Analytics API deprecated migrer vers v3.',
        'reporting-api-v2': 'Reporting API v2 active. Rapports planifiés PDF Excel white-labeling. POST /v2/reports/generate. POST /v2/reports/schedule planifier récurrent. DIFFÉRENCE vs analytics: Reporting = documents planifiés distribution.',
        'reporting-api': 'Reporting API deprecated migrer vers v2.',
        'metrics-api': 'Metrics API active. Métriques techniques ingénierie latence erreurs. POST /v1/metrics. GET /v1/metrics/{name}.',
        'event-tracking-api': 'Event Tracking API active. Comportements utilisateurs clics pages funnels analytics. POST /v1/events/track. POST /v1/events/batch. DIFFÉRENCE vs event-api: Event Tracking = comportements users analytics.',
        'ab-testing-api': 'A/B Testing API active. Expérimentations statistiques groupes contrôle variantes uplift conversion. POST /v1/experiments. POST /v1/experiments/{id}/assign variante. GET results p-value. DIFFÉRENCE vs feature-flag: AB = mesure statistique expérimentation.',
        'data-export-api': 'Data Export API active. Export massif CSV JSON Parquet. POST /v1/exports. GET /v1/exports/{jobId}/download.',
        'attribution-api': 'Attribution API active. Attribution marketing first last multi-touch. POST /v1/attribution/analyze. GET /v1/attribution/channels.',
        'cohort-api': 'Cohort API active. Cohortes LTV rétention. POST /v1/cohorts. GET /v1/cohorts/{id}/retention.',
        'customer-journey-api': 'Customer Journey API active. Parcours client omnicanal touchpoints friction. POST /v1/journey/touchpoints. GET /v1/journey/friction-points.',
        'report-api': 'Report API active. Rapport contextuel sur ressource spécifique instantané. GET /v1/report/{resourceType}/{resourceId}. POST /v1/report/batch. DIFFÉRENCE vs reporting-api: Report = rapport sur une ressource, Reporting = rapports planifiés populations.',
        'hr-api': 'HR API active. RH central organigramme départements. GET/POST /v1/hr/employees. GET /v1/hr/org-chart.',
        'employee-api': 'Employee API active. Dossiers RH contrats salaires. SCOPES hr:read hr:write hr:payroll. GET /v1/employees/{id}/salary données salariales. GET /v1/employees/{id}/contracts.',
        'payroll-api': 'Payroll API active. Paie bulletins salaire. POST /v1/payroll/run. GET /v1/payroll/{employeeId}/slips.',
        'leave-api': 'Leave API active. Congés absences soldes. POST /v1/leaves/request. GET /v1/leaves/{employeeId}/balance. PUT /v1/leaves/{id}/approve.',
        'recruitment-api': 'Recruitment API active. Offres emploi candidatures. POST /v1/jobs. POST /v1/jobs/{id}/apply. PUT /v1/candidates/{id}/status.',
        'performance-api': 'Performance API active. Évaluations OKRs KRs. POST /v1/performance/reviews. POST /v1/performance/objectives.',
        'training-api': 'Training API active. Formations compétences. GET /v1/trainings. POST /v1/trainings/{id}/enroll inscription.',
        'expense-api': 'Expense API active. Notes de frais remboursements. POST /v1/expenses. PUT /v1/expenses/{id}/approve.',
        'time-tracking-api': 'Time Tracking API active. Pointage heures supplémentaires. POST /v1/time/clock-in. POST /v1/time/clock-out.',
        'benefits-api': 'Benefits API active. Avantages sociaux mutuelle. GET /v1/benefits. POST /v1/benefits/{employeeId}/subscriptions.',
        'onboarding-api': 'Onboarding API active. Intégration nouveaux employés accès provision. POST /v1/onboarding/start. POST provision-access.',
        'org-api': 'Org API active. Entités légales filiales effectif. GET/POST /v1/organizations.',
        'health-api': 'Health API active. Santé services dépendances. GET /v1/health. GET /v1/health/services.',
        'audit-log-api': 'Audit Log API active. Journal audit traçabilité conformité. GET /v1/audit/logs. POST /v1/audit/export.',
        'gdpr-api': 'GDPR API active. Conformité RGPD droit oubli portabilité données personnelles consentements. POST /v1/gdpr/delete-request anonymisation. POST /v1/gdpr/export-request. GET/PUT /v1/gdpr/consents/{userId}.',
        'api-key-api': 'API Key API active. Clés API Kong rotation révocation. POST /v1/api-keys créer. DELETE révoquer. POST /v1/api-keys/{id}/rotate.',
        'rate-limit-api': 'Rate Limit API active. Quotas limites blacklist. GET/PUT /v1/rate-limits/{clientId}. POST /v1/rate-limits/blacklist.',
        'config-api': 'Config API active. Configuration paramètres applicatifs. GET/PUT /v1/config/{key}.',
        'feature-flag-api': 'Feature Flag API active. Déploiement progressif fonctionnalités rollout ciblage. GET/POST /v1/flags. PUT /v1/flags/{key}/rollout pourcentage. DIFFÉRENCE vs ab-testing: Feature flag = activation progressive sans mesure statistique.',
        'log-api': 'Log API active. Logs centralisés streaming. GET /v1/logs. GET /v1/logs/tail/{service}.',
        'cache-api': 'Cache API active. Cache distribué TTL invalidation. GET/PUT/DELETE /v1/cache/{key}.',
        'queue-api': 'Queue API active. Files messages DLQ. POST /v1/queues/{name}/publish. GET /v1/queues/{name}/consume.',
        'file-storage-api': 'File Storage API active. Stockage fichiers URL pré-signée. POST /v1/files/upload. POST /v1/files/presigned-url. DIFFÉRENCE vs media-api: File Storage = générique, Media = images vidéos CDN.',
        'media-api': 'Media API active. Images vidéos CDN resize. POST /v1/media/upload. POST /v1/media/{id}/resize. GET /v1/media/{id}/cdn-url. DIFFÉRENCE vs media-processing: Media = stockage CDN simple.',
        'media-processing-api': 'Media Processing API active. OCR transcodage vidéo watermark détection IA NSFW. POST /v1/media-processing/ocr extraire texte PDF. POST /v1/media-processing/transcode vidéo. POST /v1/media-processing/detect-content. DIFFÉRENCE vs media-api: Processing = transformations complexes.',
        'document-api': 'Document API active. Génération documents templates signature eIDAS. POST /v1/documents/generate. POST /v1/documents/{id}/sign.',
        'cdn-api': 'CDN API active. CDN purge règles cache. POST /v1/cdn/purge. GET /v1/cdn/stats.',
        'dns-api': 'DNS API active. Enregistrements DNS A CNAME MX. GET/POST /v1/dns/zones/{zone}/records.',
        'secret-api': 'Secret API active. Secrets vault rotation versions. POST /v1/secrets. GET /v1/secrets/{name}. POST /v1/secrets/{name}/versions.',
        'integration-api': 'Integration API active. Connecteurs ERP CRM externes. POST /v1/integrations/{name}/connect. POST /v1/integrations/{name}/sync.',
        'workflow-api': 'Workflow API active. Orchestration workflows automatisés. POST /v1/workflows/{id}/start. GET /v1/workflows/{id}/status.',
        'event-api': 'Event API active. Bus événements système microservices replay. POST /v1/events/publish. POST /v1/events/subscriptions. POST /v1/events/replay rejouer événements. DIFFÉRENCE vs event-tracking: Event = bus système microservices.',
        'ticket-api': 'Ticket API active. Tickets support client escalade. POST/GET /v1/tickets. PUT /v1/tickets/{id}/assign. PUT /v1/tickets/{id}/escalate. DIFFÉRENCE vs task: Ticket = demande support client entrante.',
        'knowledge-base-api': 'Knowledge Base API active. FAQ base connaissances recherche. GET /v1/kb/articles. GET /v1/kb/search. GET /v1/kb/suggest.',
        'survey-api': 'Survey API active. Enquêtes NPS CSAT satisfaction. POST /v1/surveys. POST /v1/surveys/{id}/send. GET /v1/surveys/nps.',
        'geolocation-api': 'Geolocation API active. GPS géocodage distance zones. POST /v1/geo/geocode. POST /v1/geo/distance.',
        'localization-api': 'Localization API active. Traductions i18n formats régionaux devises. GET /v1/l10n/translations/{lang}.',
        'translation-api': 'Translation API active. Traduction NMT automatique détection langue. POST /v1/translate. POST /v1/translate/batch.',
        'task-api': 'Task API active. Tâches internes planifiées sprint backlog dépendances. POST/GET /v1/tasks. PATCH /v1/tasks/{id}/status. GET /v1/tasks/my. DIFFÉRENCE vs ticket: Task = travail interne planifié.',
        'project-api': 'Project API active. Projets sprints jalons roadmap budget. POST/GET /v1/projects. POST /v1/projects/{id}/sprints. POST /v1/projects/{id}/milestones.',
        'team-api': 'Team API active. Équipes squads KPIs vélocité. POST/GET /v1/teams. POST /v1/teams/{id}/members. GET /v1/teams/{id}/metrics.',
        'comment-api': 'Comment API active. Commentaires annotations ressources mentions reactions. POST /v1/comments. POST /v1/comments/{id}/reactions. DIFFÉRENCE vs messaging: Comment = ancré à une ressource.',
        'contract-api': 'Contract API active. Contrats NDA SLA signature renouvellement obligations. POST/GET /v1/contracts. POST /v1/contracts/{id}/sign eIDAS. POST /v1/contracts/{id}/renew.',
        'tag-api': 'Tag API active. Tags libres multi-ressources labels. POST /v1/tags/assign. GET /v1/tags/resources.',
        'address-api': 'Address API active. Carnet adresses postales utilisateurs. GET/POST /v1/users/{userId}/addresses.',
        'contact-api': 'Contact API active. Annuaire interne entreprise vCard import. GET /v1/contacts. POST /v1/contacts/import. GET /v1/contacts/{id}/vcard.',
        'appointment-api': 'Appointment API active. Rendez-vous conseillers créneaux. GET /v1/appointments/slots. POST /v1/appointments.',
        'calendar-api': 'Calendar API active. Disponibilités internes agenda. GET /v1/calendars/{resourceId}/availability.',
        'accessibility-api': 'Accessibility API active. WCAG alt-text IA contraste. POST /v1/accessibility/analyze. POST /v1/accessibility/alt-text.',
    }
    for api_id in ALL_APIS:
        text = CATALOGUE_COMPACT.get(api_id, f'{api_id} active. API {api_id.replace("-"," ")}')
        corpus_docs_raw.append((api_id, text))


# ── Déduplication — garder uniquement la première occurrence ─────────────────
seen_ids    = set()
corpus_docs = []
corpus_ids  = []
skipped     = []

for api_id, text in corpus_docs_raw:
    if api_id not in seen_ids:
        seen_ids.add(api_id)
        corpus_docs.append(text)
        corpus_ids.append(api_id)
    else:
        skipped.append(api_id)

if skipped:
    print(f'⚠️  {len(skipped)} doublons supprimés : {skipped[:10]}')

# Vérification finale
assert len(corpus_docs) == len(set(corpus_ids)), 'Encore des doublons !'
print(f'✅ Corpus final     : {len(corpus_docs)} documents uniques')
print(f'✅ Aucun doublon    : OK')

# Diagnostique rapide
dups = {id: c for id, c in Counter(corpus_ids).items() if c > 1}
if dups:
    print(f'❌ Doublons restants : {dups}')
else:
    print(f'✅ IDs tous uniques  : OK')

# Aperçu
idx = corpus_ids.index('order-api-v4') if 'order-api-v4' in corpus_ids else 0
print(f'\n📄 Exemple ({corpus_ids[idx]}) :')
print(corpus_docs[idx][:300] + '...')


📁 Dossier trouvé : /content/apis150 (150 fichiers JSON)
⚠️  3 doublons supprimés : ['order-api', 'order-api', 'order-api']
✅ Corpus final     : 147 documents uniques
✅ Aucun doublon    : OK
✅ IDs tous uniques  : OK

📄 Exemple (ab-testing-api) :
A/B Testing API (version 1.0.0) statut active. Expérimentations et tests A/B. Variantes, assignation et analyse statistique des résultats. Equipe: Equipe Data. Domaine: Analytics & BI. ENDPOINTS: POST /v1/experiments: Créer une expérience A/B | GET /v1/experiments: Lister les expériences | GET /v1/e...


## ⚙️ Cellule 17 — Indexation ColBERT

**⏱ Durée estimée : 3-5 minutes** — le modèle colbertv2.0 (~440 MB) est téléchargé
puis les 150 documents sont encodés token par token sur GPU T4.

> Cette cellule ne s'exécute qu'une fois — l'index est réutilisé pour toutes les questions.

In [ ]:
import time
from ragatouille import RAGPretrainedModel

print('🔬 Chargement du modèle ColBERT v2.0...')
print('   Modèle : colbert-ir/colbertv2.0 (Stanford NLP, 2020)')
print('   Téléchargement ~440 MB si premier lancement...\n')

t0 = time.time()

# Chargement du modèle pré-entraîné
RAG_colbert = RAGPretrainedModel.from_pretrained('colbert-ir/colbertv2.0')

t_load = round(time.time() - t0, 1)
print(f'✅ Modèle chargé en {t_load}s')

# Indexation des 150 documents
print(f'\n⚙️  Indexation de {len(corpus_docs)} documents...')
t1 = time.time()

RAG_colbert.index(
    collection   = corpus_docs,
    document_ids = corpus_ids,
    index_name   = 'agentic4api_colbert',
    max_document_length = 512,
    split_documents     = False,
)

t_index = round(time.time() - t1, 1)
print(f'\n✅ Index ColBERT créé en {t_index}s')
print(f'   Documents indexés : {len(corpus_docs)}')
print(f'   Modèle            : colbert-ir/colbertv2.0')

# Test rapide
print('\n🔍 Test de recherche...')
results_test = RAG_colbert.search(
    query     = 'créer une commande',
    k         = 4,
    index_name= 'agentic4api_colbert'
)
print('Top 4 résultats pour "créer une commande" :')
for r in results_test:
    print(f'  #{r["rank"]} {r["document_id"]:<35} score={r["score"]:.4f}')

🔬 Chargement du modèle ColBERT v2.0...
   Modèle : colbert-ir/colbertv2.0 (Stanford NLP, 2020)
   Téléchargement ~440 MB si premier lancement...



artifact.metadata: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

...

100%|██████████| 1/1 [00:00<00:00, 5577.53it/s]

[Apr 28, 09:22:06] #> Loading codes and residuals...



100%|██████████| 1/1 [00:00<00:00, 178.77it/s]

Searcher loaded!

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: . créer une commande, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1, 27831,  2099, 16655,  3094,  2063,   102,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')



Top 4 résultats pour "créer une commande" :
  #1 order-api                           score=22.5000
  #2 pre-order-api                       score=21.3438
  #3 return-api                          score=20.7812
  #4 purchase-order-api                  score=19.5938


## 🚀 Cellule 18 — Évaluation ColBERT (100 questions)

**⏱ Durée estimée : 10-15 min** — ~0.5s par question sur GPU T4

> ColBERT fait le retrieval uniquement — il retourne les IDs des APIs les plus pertinentes.
> On mesure MRR/Recall/Precision exactement comme pour le RAG Pinecone.

In [ ]:
import time

results_colbert = []
TOTAL           = len(GOLDEN_DATASET)
TOP_K_COLBERT   = TOP_K  # même K que les autres méthodes

print(f'\n🔬 Évaluation ColBERT v2.0 — {TOTAL} questions\n')

for i, item in enumerate(GOLDEN_DATASET):
    print(f'[{i+1:3}/{TOTAL}] {item["id"]}  ({item["category"]})...', end='  ')

    t0 = time.time()

    try:
        # Recherche ColBERT — retourne les K documents les plus proches
        hits = RAG_colbert.search(
            query      = item['question'],
            k          = TOP_K_COLBERT,
            index_name = 'agentic4api_colbert'
        )
        # Extraire les IDs des APIs retrouvées (dans l'ordre de pertinence)
        retrieved = [h['document_id'] for h in hits]
        error     = None

    except Exception as e:
        retrieved = []
        error     = str(e)
        print(f'ERREUR: {e}', end='  ')

    lat = round(time.time() - t0, 3)

    p = precision_at_k(retrieved, item['expected_apis'], TOP_K_COLBERT)
    r = recall_at_k(retrieved,   item['expected_apis'], TOP_K_COLBERT)
    m = mrr(retrieved,           item['expected_apis'])

    results_colbert.append({
        **item,
        'retrieved_apis': retrieved,
        'precision': round(p, 3),
        'recall':    round(r, 3),
        'mrr':       round(m, 3),
        'latency':   lat,
        'error':     error,
    })

    status = '✅' if m > 0 or not item['expected_apis'] else '❌'
    print(f'{status}  P={p:.2f}  R={r:.2f}  MRR={m:.2f}  ({lat}s)')

print(f'\n✅ Évaluation ColBERT terminée — {TOTAL} questions')


🔬 Évaluation ColBERT v2.0 — 100 questions

[  1/100] Q001  (simple)...  ✅  P=0.25  R=1.00  MRR=0.50  (0.035s)
[  2/100] Q002  (simple)...  ✅  P=0.50  R=1.00  MRR=1.00  (0.028s)
[  3/100] Q003  (simple)...  ✅  P=0.50  R=1.00  MRR=1.00  (0.029s)
[  4/100] Q004  (simple)...  ✅  P=0.50  R=1.00  MRR=1.00  (0.032s)
[  5/100] Q005  (simple)...  ✅  P=0.25  R=1.00  MRR=1.00  (0.03s)
[  6/100] Q006  (simple)...  ✅  P=0.25  R=1.00  MRR=1.00  (0.028s)
[  7/100] Q007  (simple)...  ✅  P=0.25  R=1.00  MRR=1.00  (0.029s)
[  8/100] Q008  (simple)...  ✅  P=0.75  R=1.00  MRR=1.00  (0.028s)
[  9/100] Q009  (simple)...  ✅  P=0.50  R=1.00  MRR=1.00  (0.029s)
[ 10/100] Q010  (simple)...  ✅  P=0.25  R=1.00  MRR=0.50  (0.028s)
[ 11/100] Q011  (simple)...  ✅  P=0.75  R=1.00  MRR=1.00  (0.028s)
[ 12/100] Q012  (simple)...  ✅  P=0.25  R=1.00  MRR=1.00  (0.034s)
[ 13/100] Q013  (simple)...  ✅  P=0.25  R=1.00  MRR=1.00  (0.027s)
[ 14/100] Q014  (simple)...  ✅  P=0.25  R=1.00  MRR=0.50  (0.027s)
[ 15/100] Q015  (si

## 📊 Cellule 19 — Métriques ColBERT

In [ ]:
import pandas as pd

n_cb = len(results_colbert)

global_metrics_colbert = {
    'Agent':             'ColBERT v2.0 (Late Interaction)',
    'Questions':          n_cb,
    'Precision@K':        round(sum(r['precision'] for r in results_colbert) / n_cb, 3),
    'Recall@K':           round(sum(r['recall']    for r in results_colbert) / n_cb, 3),
    'MRR':                round(sum(r['mrr']       for r in results_colbert) / n_cb, 3),
    'Latence moy. (s)':   round(sum(r['latency']   for r in results_colbert) / n_cb, 3),
    'Erreurs':            sum(1 for r in results_colbert if r['error']),
    'Questions réussies (MRR>0)': sum(1 for r in results_colbert if r['mrr'] > 0 or not r['expected_apis']),
}

print('\n' + '='*60)
print('  RÉSULTATS — ColBERT v2.0')
print('='*60)
for k, v in global_metrics_colbert.items():
    print(f'  {k:<30} : {v}')

# Par catégorie
cats_cb = sorted(set(r['category'] for r in results_colbert))
rows_cat_cb = []
for cat in cats_cb:
    subset = [r for r in results_colbert if r['category'] == cat]
    rows_cat_cb.append({
        'Catégorie':   cat,
        'N':           len(subset),
        'Precision@K': round(sum(r['precision'] for r in subset) / len(subset), 3),
        'Recall@K':    round(sum(r['recall']    for r in subset) / len(subset), 3),
        'MRR':         round(sum(r['mrr']       for r in subset) / len(subset), 3),
        'Latence (s)': round(sum(r['latency']   for r in subset) / len(subset), 3),
    })

df_cat_cb = pd.DataFrame(rows_cat_cb).set_index('Catégorie')
print('\n' + '='*60)
print('  PAR CATÉGORIE — ColBERT')
print('='*60)
print(df_cat_cb.to_string())

# Par difficulté
diffs_cb = sorted(set(r['difficulty'] for r in results_colbert))
rows_diff_cb = []
for diff in diffs_cb:
    subset = [r for r in results_colbert if r['difficulty'] == diff]
    rows_diff_cb.append({
        'Difficulté':  diff,
        'N':           len(subset),
        'MRR':         round(sum(r['mrr']    for r in subset) / len(subset), 3),
        'Recall@K':    round(sum(r['recall'] for r in subset) / len(subset), 3),
    })

df_diff_cb = pd.DataFrame(rows_diff_cb).set_index('Difficulté')
print('\n' + '='*60)
print('  PAR DIFFICULTÉ — ColBERT')
print('='*60)
print(df_diff_cb.to_string())

# Questions ratées
failed_cb = [r for r in results_colbert if r['mrr'] == 0 and r['expected_apis']]
print(f'\n  QUESTIONS RATÉES ColBERT : {len(failed_cb)}/{n_cb}')
for r in failed_cb:
    print(f'  {r["id"]} | {r["category"]:<20} | attendu: {r["expected_apis"]}')
    print(f'           obtenu: {r["retrieved_apis"]}')


  RÉSULTATS — ColBERT v2.0
  Agent                          : ColBERT v2.0 (Late Interaction)
  Questions                      : 100
  Precision@K                    : 0.355
  Recall@K                       : 0.85
  MRR                            : 0.862
  Latence moy. (s)               : 0.03
  Erreurs                        : 0
  Questions réussies (MRR>0)     : 93

  PAR CATÉGORIE — ColBERT
                   N  Precision@K  Recall@K    MRR  Latence (s)
Catégorie                                                      
authorization      5        0.300     1.000  0.900        0.028
breaking_change    4        0.375     0.750  0.750        0.031
endpoint_precise  10        0.300     0.900  0.900        0.029
faux_positif      20        0.412     0.975  0.925        0.037
multi_api         15        0.417     0.443  0.611        0.028
negative          10        0.025     0.900  0.900        0.029
simple            15        0.383     1.000  0.900        0.029
use_case          10      

## ⚖️ Cellule 20 — Comparaison Finale Toutes Méthodes

**Le tableau central du Point 3 du mémoire** — Dense RAG vs ColBERT vs Sans RAG.

> Note : Si tu as aussi les résultats BM25/TF-IDF, ils s'ajoutent facilement ici.

In [ ]:
import pandas as pd

# ── Tableau comparatif toutes méthodes ───────────────────────────────────────
methods = []

# Dense RAG (résultats existants)
methods.append({
    'Méthode':        '🔵 Dense RAG (Gemini + Pinecone)',
    'Type':           'Sémantique',
    'MRR':            global_metrics['MRR'],
    'Recall@K':       global_metrics['Recall@K'],
    'Precision@K':    global_metrics['Precision@K'],
    'Latence (s)':    global_metrics['Latence moy. (s)'],
    'Tokens/question': '~2 000',
    'Infrastructure': 'n8n + Pinecone Cloud',
})

# ColBERT
methods.append({
    'Méthode':        '🔬 ColBERT v2.0 (Late Interaction)',
    'Type':           'Sémantique avancé',
    'MRR':            global_metrics_colbert['MRR'],
    'Recall@K':       global_metrics_colbert['Recall@K'],
    'Precision@K':    global_metrics_colbert['Precision@K'],
    'Latence (s)':    global_metrics_colbert['Latence moy. (s)'],
    'Tokens/question': 'N/A (local)',
    'Infrastructure': 'GPU T4 local',
})

# Sans RAG
methods.append({
    'Méthode':        '🔴 Sans RAG (Prompt Stuffing)',
    'Type':           'LLM direct',
    'MRR':            global_metrics_no_rag['MRR'],
    'Recall@K':       global_metrics_no_rag['Recall@K'],
    'Precision@K':    global_metrics_no_rag['Precision@K'],
    'Latence (s)':    global_metrics_no_rag['Latence moy. (s)'],
    'Tokens/question': '~60 000',
    'Infrastructure': 'n8n Cloud direct',
})

df_final = pd.DataFrame(methods).set_index('Méthode')
print('\n' + '='*75)
print('  COMPARAISON FINALE — Dense RAG vs ColBERT vs Sans RAG')
print('='*75)
print(df_final[['Type','MRR','Recall@K','Precision@K','Latence (s)']].to_string())

# ── Par catégorie ─────────────────────────────────────────────────────────────
cats_all = sorted(set(r['category'] for r in results))
rows_all_cat = []
for cat in cats_all:
    rag_s  = [r for r in results         if r['category'] == cat]
    cb_s   = [r for r in results_colbert  if r['category'] == cat]
    nr_s   = [r for r in results_no_rag   if r['category'] == cat]
    mrr_rag = round(sum(r['mrr'] for r in rag_s) / len(rag_s), 3)
    mrr_cb  = round(sum(r['mrr'] for r in cb_s)  / len(cb_s),  3) if cb_s else 0
    mrr_nr  = round(sum(r['mrr'] for r in nr_s)  / len(nr_s),  3) if nr_s else 0
    best    = max([(mrr_rag,'RAG'),(mrr_cb,'ColBERT'),(mrr_nr,'Sans RAG')], key=lambda x:x[0])
    rows_all_cat.append({
        'Catégorie':    cat,
        'N':            len(rag_s),
        'Dense RAG':    mrr_rag,
        'ColBERT':      mrr_cb,
        'Sans RAG':     mrr_nr,
        'Meilleur':     best[1],
    })

df_all_cat = pd.DataFrame(rows_all_cat).set_index('Catégorie')
print('\n' + '='*75)
print('  MRR PAR CATÉGORIE — toutes méthodes')
print('='*75)
print(df_all_cat.to_string())

# ── Synthèse pour le mémoire ──────────────────────────────────────────────────
print('\n' + '='*75)
print('  SYNTHÈSE POUR LE MÉMOIRE')
print('='*75)
mrr_rag = global_metrics['MRR']
mrr_cb  = global_metrics_colbert['MRR']
mrr_nr  = global_metrics_no_rag['MRR']

print(f'  Dense RAG  MRR={mrr_rag} | Latence={global_metrics["Latence moy. (s)"]}s | Tokens=~2 000 | Cloud ✅')
print(f'  ColBERT    MRR={mrr_cb}  | Latence={global_metrics_colbert["Latence moy. (s)"]}s | GPU requis ⚠️')
print(f'  Sans RAG   MRR={mrr_nr} | Latence={global_metrics_no_rag["Latence moy. (s)"]}s | Tokens=~60 000 ❌ scalable')

if mrr_cb > mrr_rag:
    delta = round(mrr_cb - mrr_rag, 3)
    print(f'\n  ColBERT > Dense RAG de +{delta} MRR.')
    print(f'  Mais ColBERT nécessite GPU local — incompatible avec architecture n8n cloud.')
    print(f'  Conclusion : Dense RAG = meilleur compromis performance/déployabilité.')
elif mrr_rag >= mrr_cb:
    print(f'\n  Dense RAG ≥ ColBERT sur ce corpus.')
    print(f'  ColBERT apporte peu de valeur supplémentaire pour ce cas d\'usage.')

NameError: name 'global_metrics' is not defined

## 📈 Cellule 21 — Dashboard Final Toutes Méthodes

Visualisation comparative pour le mémoire.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

BG      = '#f8f9fa'
C_RAG   = '#378ADD'
C_CB    = '#534AB7'
C_NORAG = '#E24B4A'
colors  = [C_RAG, C_CB, C_NORAG]
labels  = ['Dense RAG\n(Gemini+Pinecone)', 'ColBERT v2.0\n(Late Interaction)', 'Sans RAG\n(Prompt Stuffing)']

mrr_vals   = [global_metrics['MRR'],          global_metrics_colbert['MRR'],         global_metrics_no_rag['MRR']]
rec_vals   = [global_metrics['Recall@K'],      global_metrics_colbert['Recall@K'],    global_metrics_no_rag['Recall@K']]
prec_vals  = [global_metrics['Precision@K'],   global_metrics_colbert['Precision@K'], global_metrics_no_rag['Precision@K']]
lat_vals   = [global_metrics['Latence moy. (s)'], global_metrics_colbert['Latence moy. (s)'], global_metrics_no_rag['Latence moy. (s)']]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor(BG)
fig.suptitle('Agentic4API — Comparaison Méthodes de Retrieval\nDense RAG vs ColBERT vs Sans RAG · 100 questions · 150 APIs',
             fontsize=14, fontweight='bold', color='#222', y=1.01)

def bar_chart(ax, vals, title, ylabel, highlight_max=True):
    x = np.arange(len(labels))
    bars = ax.bar(x, vals, color=colors, alpha=0.87, width=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold', color='#333')
    ax.set_facecolor(BG)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top','right']].set_visible(False)
    ax.set_ylim(0, max(vals) * 1.2)
    max_val = max(vals)
    for bar, val in zip(bars, vals):
        color_val = '#185FA5' if bar.get_facecolor()[:3] == (0.216, 0.478, 0.678) else '#333'
        weight = 'bold' if val == max_val and highlight_max else 'normal'
        ax.text(bar.get_x() + bar.get_width()/2, val + max_val*0.02,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=11, fontweight=weight, color='#222')
        if val == max_val and highlight_max:
            ax.annotate('★', xy=(bar.get_x() + bar.get_width()/2, val + max_val*0.09),
                       ha='center', fontsize=14, color='#BA7517')

bar_chart(axes[0,0], mrr_vals,  'MRR (Mean Reciprocal Rank)', 'MRR')
bar_chart(axes[0,1], rec_vals,  'Recall@K',                   'Recall@K')
bar_chart(axes[1,0], prec_vals, 'Precision@K',                'Precision@K')

# Latence — ici le min est le meilleur
x = np.arange(len(labels))
bars_lat = axes[1,1].bar(x, lat_vals, color=colors, alpha=0.87, width=0.5)
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(labels, fontsize=10)
axes[1,1].set_ylabel('Secondes', fontsize=10)
axes[1,1].set_title('Latence moyenne (↓ = meilleur)', fontsize=12, fontweight='bold', color='#333')
axes[1,1].set_facecolor(BG)
axes[1,1].grid(axis='y', alpha=0.3)
axes[1,1].spines[['top','right']].set_visible(False)
min_lat = min(lat_vals)
for bar, val in zip(bars_lat, lat_vals):
    weight = 'bold' if val == min_lat else 'normal'
    axes[1,1].text(bar.get_x() + bar.get_width()/2, val + max(lat_vals)*0.02,
                   f'{val:.2f}s', ha='center', va='bottom', fontsize=11, fontweight=weight, color='#222')
    if val == min_lat:
        axes[1,1].annotate('★', xy=(bar.get_x() + bar.get_width()/2, val + max(lat_vals)*0.1),
                           ha='center', fontsize=14, color='#BA7517')

plt.tight_layout()
plt.savefig('colbert_vs_rag_dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✅ Dashboard sauvegardé → colbert_vs_rag_dashboard.png')

## 💾 Cellule 22 — Sauvegarde Résultats ColBERT

In [ ]:
import json

# Ajouter ColBERT aux résultats existants
try:
    with open('agentic4api_full_evaluation.json', 'r', encoding='utf-8') as f:
        output_final = json.load(f)
except FileNotFoundError:
    output_final = {}

output_final['colbert'] = {
    'agent':           'ColBERT v2.0 (Late Interaction)',
    'model':           'colbert-ir/colbertv2.0',
    'reference':       'Khattab & Zaharia 2020, Stanford NLP',
    'total_questions':  n_cb,
    'global_metrics':   global_metrics_colbert,
    'by_category':      rows_cat_cb,
    'by_difficulty':    rows_diff_cb,
    'raw_results':      results_colbert,
}

output_final['final_comparison'] = {
    'dense_rag_mrr':    global_metrics['MRR'],
    'colbert_mrr':      global_metrics_colbert['MRR'],
    'no_rag_mrr':       global_metrics_no_rag['MRR'],
    'best_mrr_method':  max([
        ('Dense RAG', global_metrics['MRR']),
        ('ColBERT',   global_metrics_colbert['MRR']),
        ('Sans RAG',  global_metrics_no_rag['MRR']),
    ], key=lambda x: x[1])[0],
    'note': 'Sans RAG gagne en MRR sur 150 APIs (context window Gemini 1M tokens). ColBERT nécessite GPU local. Dense RAG = meilleur compromis déployabilité/performance.',
}

with open('agentic4api_full_evaluation.json', 'w', encoding='utf-8') as f:
    json.dump(output_final, f, ensure_ascii=False, indent=2)

print('✅ Résultats complets sauvegardés → agentic4api_full_evaluation.json')
print(f'\n  RÉSUMÉ FINAL COMPLET :')
print(f'  Dense RAG  : MRR={global_metrics["MRR"]}  | Latence={global_metrics["Latence moy. (s)"]}s')
print(f'  ColBERT    : MRR={global_metrics_colbert["MRR"]}  | Latence={global_metrics_colbert["Latence moy. (s)"]}s')
print(f'  Sans RAG   : MRR={global_metrics_no_rag["MRR"]} | Latence={global_metrics_no_rag["Latence moy. (s)"]}s')

try:
    from google.colab import files
    files.download('agentic4api_full_evaluation.json')
    files.download('colbert_vs_rag_dashboard.png')
    print('\n✅ Fichiers téléchargés')
except ImportError:
    print('\n✅ Fichiers sauvegardés localement')